# Ciencia de Datos — RDA 2
## Práctica integradora: pipeline de extracción e integración de datos

**Estudiante: Joaquín Bermeo**

Basado en `RDA2_Extraccion_de_datos_con_Python.ipynb`. Se excluyen Parquet y la bitácora `pipeline.log`, según la indicación del docente.

**Escenario:** TecnoAndes Cía. Ltda., cadena de tiendas de tecnología &nbsp;|&nbsp; **Bibliotecas:** Pandas, JSON, SQLite3, Pathlib, Logging

---

### Objetivo de la práctica

Construir un **pipeline reproducible** que extraiga datos de varias fuentes heterogéneas, los estandarice, los integre mediante identificadores y nombres propios, valide el resultado y lo guarde como un **nuevo conjunto de datos listo para el Análisis Exploratorio de Datos (EDA)**.

### Resultados esperados

Al finalizar, el estudiante será capaz de:

1. Extraer datos desde archivos CSV con convenciones distintas, archivos JSON con metadatos, un libro de Excel y una base de datos SQLite.
2. Diagnosticar problemas de calidad antes de transformar los datos.
3. Construir claves de unión a partir de identificadores con formatos diferentes y de nombres propios escritos de forma inconsistente.
4. Integrar varias fuentes sin perder ni multiplicar registros.
5. Aplicar reglas de negocio documentadas y separar los registros rechazados.
6. Guardar el resultado con su diccionario de datos y su reporte de calidad.
7. Ejecutar todo el proceso con una sola función y comprobar que es reproducible.

---
## 1. Contexto del caso

### 1.1. Requerimiento de la gerencia

> **De:** Gerencia Comercial — TecnoAndes Cía. Ltda.
> **Para:** Equipo de Ciencia de Datos
> **Asunto:** Consolidado de ventas enero 2025 – junio 2026
>
> Necesitamos analizar el desempeño comercial de nuestras tres sucursales (Ambato, Latacunga y Píllaro). El problema es que la información está dispersa: cada sucursal exporta sus ventas en un formato distinto, el catálogo sale del ERP, los clientes del CRM, la nómina de vendedores la maneja Talento Humano en Excel y los costos e inventario están en la base de datos de bodega.
>
> Les pedimos un **único conjunto de datos de ventas**, confiable y documentado, que el equipo de análisis pueda usar directamente. El proceso debe poder repetirse cada mes con los nuevos archivos sin hacer cambios manuales.

### 1.2. Fuentes de datos disponibles

| N.º | Sistema de origen | Archivo | Formato | Contenido |
|---|---|---|---|---|
| 1 | Punto de venta Ambato | `ventas/ventas_ambato.csv` | CSV | Líneas de venta |
| 2 | Punto de venta Latacunga | `ventas/ventas_latacunga.csv` | CSV (sistema antiguo) | Líneas de venta |
| 3 | Punto de venta Píllaro | `ventas/ventas_pillaro.csv` | CSV (exportado de hoja de cálculo) | Líneas de venta |
| 4 | ERP | `erp/catalogo_productos.json` | JSON con metadatos | Categorías y productos |
| 5 | CRM | `crm/clientes_lote_1.json` ... `_3.json` | JSON con metadatos, por lotes | Clientes |
| 6 | Talento Humano | `rrhh/nomina_vendedores.xlsx` | Excel | Vendedores y comisiones |
| 7 | Bodega | `inventario/inventario.db` | SQLite | Sucursales, costos y stock |

### 1.3. Relaciones entre fuentes

```
                         ┌──────────────────────┐
                         │  CATÁLOGO (JSON)     │
                         │  sku: "TA-0001"      │
                         └──────────▲───────────┘
                                    │ id_producto (formatos distintos)
┌──────────────────┐     ┌──────────┴───────────┐     ┌──────────────────────┐
│ CLIENTES (JSON)  │     │   VENTAS (3 CSV)     │     │ VENDEDORES (Excel)   │
│ id: "C0001"      ◄─────┤   tabla de hechos    ├─────► nombre completo      │
└──────────────────┘     │                      │     └──────────────────────┘
   id_cliente            └──────────┬───────────┘       nombre propio
   (formatos distintos)             │ id_sucursal, id_producto
                         ┌──────────▼───────────┐
                         │ INVENTARIO (SQLite)  │
                         │ sucursales, costos,  │
                         │ stock                │
                         └──────────────────────┘
```

- **Ventas – Catálogo:** por identificador de producto, escrito como `TA-0001`, `ta0001` o `1` según la sucursal.
- **Ventas – Clientes:** por identificador de cliente, escrito como `C0001` o `CLI-1`.
- **Ventas – Vendedores:** **no existe identificador**; la unión se hace por el **nombre propio** del vendedor.
- **Ventas – Inventario:** por sucursal y producto.

### 1.4. Reglas de negocio acordadas con la gerencia

Toda decisión sobre los datos debe estar respaldada por una regla. Estas son las acordadas:

| Código | Regla |
|---|---|
| R1 | Las filas completamente duplicadas se eliminan. |
| R2 | Las líneas con cantidad menor o igual a cero se **rechazan** (no son ventas válidas). |
| R3 | Las líneas cuyo producto no existe en el catálogo se **rechazan**. |
| R4 | Las ventas sin cliente se asignan al cliente genérico `C0000` "CONSUMIDOR FINAL". |
| R5 | Las ventas con un cliente que no existe en el CRM se conservan con el cliente "NO REGISTRADO". |
| R6 | Las ventas sin vendedor se conservan con el vendedor "SIN ASIGNAR". |
| R7 | Las cantidades mayores a 20 unidades se conservan, pero se marcan como venta atípica. |
| R8 | Si un cliente aparece en varios lotes del CRM, se conserva la versión con la fecha de actualización más reciente. |
| R9 | Si un producto aparece repetido en el catálogo, se conserva la primera aparición. |
| R10 | Del inventario se usa únicamente el corte de stock más reciente; los stocks negativos se consideran inconsistentes y se reemplazan por nulo. |
| R11 | Los costos faltantes **no se imputan**; el margen queda nulo y se decidirá su tratamiento en el EDA. |
| R12 | Los registros rechazados no se eliminan sin dejar rastro: se guardan en un archivo aparte con el motivo. |

### 1.5. Entregables del pipeline

| Archivo | Ubicación | Descripción |
|---|---|---|
| `ventas_consolidadas.csv` | `data/processed/` | Dataset final, una fila por línea de venta |
| `diccionario_datos.csv` | `data/processed/` | Descripción de cada columna |
| `reporte_calidad.csv` | `data/processed/` | Resultado de las reglas de validación |
| `registros_rechazados.csv` | `data/processed/` | Registros excluidos y su motivo |
| Fuentes estandarizadas | `data/interim/` | Cada fuente limpia, antes de la integración |

### 1.6. Instrucciones de trabajo

1. Descomprima el archivo de datos de modo que la carpeta `data/` quede **en la carpeta superior a Notebook/**.
2. Ejecute las celdas en orden. Las celdas marcadas como **Ejercicio** contienen funciones por completar: reemplace la línea `raise NotImplementedError(...)` por su código.
3. Después de cada ejercicio hay una **celda de verificación**. Cada comprobación muestra `[CORRECTO]` o `[REVISAR]`. No modifique las celdas de verificación.
4. No modifique los archivos de `data/raw/`: son la evidencia original.
5. Documente sus decisiones en las celdas de respuesta.

### 1.7. Etapas del pipeline

```
 data/raw ──> 1. Extraer ──> 2. Diagnosticar ──> 3. Estandarizar ──> data/interim
                                                        │
 data/processed <── 7. Guardar <── 6. Validar <── 5. Derivar <── 4. Integrar
```

---
## 2. Configuración del proyecto

Un pipeline reproducible **no tiene valores escritos dentro de las funciones**: rutas, formatos y parámetros se concentran en un único lugar. Si el próximo mes cambia una ruta o un umbral, solo se modifica la configuración.

### 2.1. Bibliotecas

In [1]:
import json
import logging
import re
import sqlite3
import hashlib
import unicodedata
from contextlib import closing
from datetime import datetime
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("Pandas:", pd.__version__)

Pandas: 3.0.6


### 2.2. Inventario de fuentes crudas

Antes de configurar nada, se verifica que los archivos existan.

In [2]:
def localizar_proyecto():
    for carpeta in (Path.cwd(), *Path.cwd().parents):
        candidatos = [carpeta, carpeta / "Clase11_11_2026",
                      carpeta / "NOTEBOOKS/Clase11_11_2026",
                      carpeta / "ciencia-de-datos/NOTEBOOKS/Clase11_11_2026"]
        for candidato in candidatos:
            if (candidato / "data/raw/ventas/ventas_ambato.csv").is_file():
                return candidato.resolve()
    raise FileNotFoundError("No se encontró Clase11_11_2026/data/raw.")

BASE_DIR = localizar_proyecto()
RAW_DIR = BASE_DIR / "data/raw"
inventario_archivos = pd.DataFrame([
    {"archivo": str(p.relative_to(RAW_DIR)), "tamano_kb": round(p.stat().st_size / 1024, 1)}
    for p in sorted(RAW_DIR.rglob("*")) if p.is_file()
])
inventario_archivos

,archivo,tamano_kb
0,crm\clientes_lote_1.json,41.8
1,crm\clientes_lote_2.json,46.1
2,crm\clientes_lote_3.json,46.1
3,erp\catalogo_productos.json,20.3
4,inventario\inventario.db,24.0
5,rrhh\nomina_vendedores.xlsx,5.9
6,ventas\ventas_ambato.csv,46.3
7,ventas\ventas_latacunga.csv,35.7
8,ventas\ventas_pillaro.csv,34.7


### 2.3. Reconocimiento de las fuentes de ventas

Antes de leer un archivo con Pandas hay que **mirarlo como texto**. Esta revisión define cómo se configurará la lectura de cada sucursal.

In [3]:
def vistazo(ruta: Path, lineas: int = 4, encoding: str = "utf-8") -> None:
    """Muestra las primeras líneas de un archivo de texto tal como están guardadas."""
    print(f"--- {ruta.name} ({encoding}) ---")
    with open(ruta, "r", encoding=encoding, errors="replace") as f:
        for _ in range(lineas):
            print(f.readline().rstrip())
    print()


for archivo in sorted((RAW_DIR / "ventas").glob("*.csv")):
    vistazo(archivo)

--- ventas_ambato.csv (utf-8) ---
id_venta,fecha,cod_producto,id_cliente,vendedor,cantidad,precio_unitario,descuento
AMB-00001,2025-01-01,TA-0024,C0216,  maría josé torres ,1,9.09,0.0
AMB-00002,2025-01-01,TA-0025,C0137,  sofía paredes ,1,91.73,0.15
AMB-00003,2025-01-02,TA-0031,C0177,SOFIA PAREDES,2,57.61,0.0

--- ventas_latacunga.csv (utf-8) ---
Nro_Venta;Fecha_Venta;Codigo;Cliente_ID;Vendedor;Cant;PVP;Desc
1;02/01/2025;ta0060;CLI-76;Diego Ch�vez;1;139,51;0,0
2;03/01/2025;ta0036;CLI-86;Diego Ch�vez;1;41,2;0,1
3;06/01/2025;ta0022;CLI-62;Diego Ch�vez;1;67,34;0,0

--- ventas_pillaro.csv (utf-8) ---
id_venta,fecha_hora,producto_id,cliente,vendedor,unidades,precio,descuento
PIL-00001,2025/01/04 09:20,33,C0168,Gabriela Ortiz,1,"$85,42",10%
PIL-00002,2025/01/04 10:11,39,C0051,  gabriela ortiz ,1,"$85,66",0%
PIL-00003,2025/01/04 15:21,37,C0063,GABRIELA ORTIZ,2,"$42,94",5%



En `ventas_latacunga.csv` aparece el carácter `�` en lugar de las tildes: el archivo **no** está en UTF-8. Se prueba con `latin-1`:

In [4]:
vistazo(RAW_DIR / "ventas" / "ventas_latacunga.csv", encoding="latin-1")

--- ventas_latacunga.csv (latin-1) ---
Nro_Venta;Fecha_Venta;Codigo;Cliente_ID;Vendedor;Cant;PVP;Desc
1;02/01/2025;ta0060;CLI-76;Diego Chávez;1;139,51;0,0
2;03/01/2025;ta0036;CLI-86;Diego Chávez;1;41,2;0,1
3;06/01/2025;ta0022;CLI-62;Diego Chávez;1;67,34;0,0



**Reconocimiento de las fuentes de ventas**

| Aspecto | Ambato | Latacunga | Píllaro |
|---|---|---|---|
| Separador | Coma | Punto y coma | Coma y montos entre comillas |
| Codificación | UTF-8 | Latin-1 | UTF-8 |
| Separador decimal | Punto | Coma | Coma |
| Formato de fecha | `%Y-%m-%d` | `%d/%m/%Y` | `%Y/%m/%d %H:%M` |
| Formato del código de producto | `TA-0024` | `ta0060` | `33` |
| Formato del cliente | `C0216` | `CLI-76` | `C0168` |
| Representación del descuento | Proporción con punto: `0.15` | Proporción con coma: `0,1` | Porcentaje: `10%` |
| Representación de valores vacíos | Campo vacío | `N/A` | `-` |

### 2.4. Configuración central

El diccionario `CONFIG` concentra todas las decisiones de lectura. La sección `fuentes_ventas` indica, para cada sucursal, cómo leer el archivo y cómo **renombrar sus columnas a un esquema común**:

`id_venta, fecha, cod_producto, id_cliente, vendedor, cantidad, precio_unitario, descuento`

#### Ejercicio 1. Configuración de las fuentes de ventas

La configuración de Ambato está completa. Complete la de **Latacunga** y **Píllaro** usando lo observado en el reconocimiento. Los formatos de fecha usan la notación de `strftime` (`%d` día, `%m` mes, `%Y` año, `%H` hora, `%M` minuto).

In [5]:
CONFIG = {
    "semilla": 2026,
    "periodo": {"inicio": "2025-01-01", "fin": "2026-06-30"},
    "rutas": {
        "raw": RAW_DIR,
        "interim": BASE_DIR / "data/interim",
        "processed": BASE_DIR / "data/processed",
    },
    "fuentes_ventas": {
        "ambato": {
            "archivo": "ventas/ventas_ambato.csv",
            "id_sucursal": 1,
            "lectura": {"sep": ",", "encoding": "utf-8"},
            "decimal": ".",
            "formato_fecha": "%Y-%m-%d",
            "prefijo_id": None,
            "columnas": {
                "id_venta": "id_venta", "fecha": "fecha", "cod_producto": "cod_producto",
                "id_cliente": "id_cliente", "vendedor": "vendedor", "cantidad": "cantidad",
                "precio_unitario": "precio_unitario", "descuento": "descuento",
            },
        },
        "latacunga": {
            "archivo": "ventas/ventas_latacunga.csv",
            "id_sucursal": 2,
            "lectura": {"sep": ";", "encoding": "latin-1"},
            "decimal": ",",
            "formato_fecha": "%d/%m/%Y",
            "prefijo_id": "LAT-",
            "columnas": {
                "Nro_Venta": "id_venta", "Fecha_Venta": "fecha", "Codigo": "cod_producto",
                "Cliente_ID": "id_cliente", "Vendedor": "vendedor", "Cant": "cantidad",
                "PVP": "precio_unitario", "Desc": "descuento",
            },
        },
        "pillaro": {
            "archivo": "ventas/ventas_pillaro.csv",
            "id_sucursal": 3,
            "lectura": {"sep": ",", "encoding": "utf-8"},
            "decimal": ",",
            "formato_fecha": "%Y/%m/%d %H:%M",
            "prefijo_id": None,
            "columnas": {
                "id_venta": "id_venta", "fecha_hora": "fecha", "producto_id": "cod_producto",
                "cliente": "id_cliente", "vendedor": "vendedor", "unidades": "cantidad",
                "precio": "precio_unitario", "descuento": "descuento",
            },
        },
    },
    "catalogo": "erp/catalogo_productos.json",
    "clientes_patron": "crm/clientes_lote_*.json",
    "vendedores": {"archivo": "rrhh/nomina_vendedores.xlsx", "hoja": "Personal", "fila_encabezado": 3},
    "inventario": "inventario/inventario.db",
    "valores_nulos": ["", "N/A", "NA", "-", "null", "None"],
    "cliente_consumidor_final": "C0000",
    "umbral_cantidad_atipica": 20,
    "salida": {"nombre": "ventas_consolidadas"},
}

COLUMNAS_ESTANDAR = ["id_venta", "fecha", "cod_producto", "id_cliente",
                     "vendedor", "cantidad", "precio_unitario", "descuento"]

### 2.5. Utilidades del pipeline

Se muestran mensajes en pantalla, sin crear una bitácora de ejecución. `perfilar` permite
revisar tipos, nulos, valores únicos y duplicados antes de transformar las fuentes.

In [6]:
def configurar_logger(config: dict) -> logging.Logger:
    logger = logging.getLogger("pipeline_tecnoandes_resuelto")
    logger.setLevel(logging.INFO)
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
        handler.close()
    pantalla = logging.StreamHandler()
    pantalla.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(pantalla)
    logger.propagate = False
    return logger


def ejecutar_paso(nombre: str, funcion, *args, **kwargs):
    try:
        return funcion(*args, **kwargs)
    except Exception:
        logger.exception("Falló la etapa: %s", nombre)
        raise


def verificar(descripcion: str, condicion) -> bool:
    """Evalúa una condición (función sin argumentos) y muestra el resultado."""
    try:
        correcto = bool(condicion())
    except Exception:
        correcto = False
    print(("[CORRECTO] " if correcto else "[REVISAR]  ") + descripcion)
    return correcto


def perfilar(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """Resumen de calidad: tipo, nulos, valores únicos y un ejemplo por columna."""
    texto = df.astype(str)
    print(f"{nombre}: {df.shape[0]} filas x {df.shape[1]} columnas | "
          f"filas duplicadas: {texto.duplicated().sum()}")
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isnull().sum(),
        "unicos": texto.nunique(),
        "ejemplo": df.iloc[0] if len(df) else None,
    })


logger = configurar_logger(CONFIG)
logger.info("Configuración cargada")

INFO | Configuración cargada


**Verificación del Ejercicio 1**

In [7]:
for suc, cfg in CONFIG["fuentes_ventas"].items():
    verificar(f"{suc}: las columnas se mapean al esquema estándar",
              lambda cfg=cfg: sorted(cfg["columnas"].values()) == sorted(COLUMNAS_ESTANDAR))
    verificar(f"{suc}: separador, codificación, decimal y formato de fecha definidos",
              lambda cfg=cfg: all([cfg["lectura"]["sep"], cfg["lectura"]["encoding"],
                                   cfg["decimal"], cfg["formato_fecha"]]))
verificar("Latacunga usa latin-1 y punto y coma",
          lambda: CONFIG["fuentes_ventas"]["latacunga"]["lectura"] == {"sep": ";", "encoding": "latin-1"})
verificar("Formato de fecha de Píllaro correcto",
          lambda: datetime.strptime("2025/01/04 09:20", CONFIG["fuentes_ventas"]["pillaro"]["formato_fecha"]))

[CORRECTO] ambato: las columnas se mapean al esquema estándar
[CORRECTO] ambato: separador, codificación, decimal y formato de fecha definidos
[CORRECTO] latacunga: las columnas se mapean al esquema estándar
[CORRECTO] latacunga: separador, codificación, decimal y formato de fecha definidos
[CORRECTO] pillaro: las columnas se mapean al esquema estándar
[CORRECTO] pillaro: separador, codificación, decimal y formato de fecha definidos
[CORRECTO] Latacunga usa latin-1 y punto y coma
[CORRECTO] Formato de fecha de Píllaro correcto


True

---
## 3. Extracción

**Principio:** en esta etapa los datos se leen **sin corregirlos**. El objetivo es obtenerlos de forma completa y fiel a la fuente. Por eso las ventas se leen con `dtype=str`: así ningún valor se convierte de forma automática (por ejemplo, `0001` no pierde sus ceros y `$85,42` no genera errores). Las conversiones se harán, de forma controlada, en la etapa de estandarización.

### 3.1. Ventas: tres archivos, un esquema

#### Ejercicio 2. Extraer las ventas de las tres sucursales

Implemente `extraer_ventas(config)` para que, por cada sucursal de `config["fuentes_ventas"]`:

1. Lea el CSV con `pd.read_csv`, usando los parámetros de `lectura`, `dtype=str` y `na_values=config["valores_nulos"]`.
2. Renombre las columnas con el diccionario `columnas`.
3. Agregue las columnas `fuente` (nombre de la sucursal, por ejemplo `"ambato"`) e `id_sucursal`.

Finalmente, concatene las tres tablas en un solo DataFrame con `ignore_index=True`.

In [8]:
def extraer_ventas(config: dict) -> pd.DataFrame:
    partes = []
    for fuente, cfg in config["fuentes_ventas"].items():
        ruta = config["rutas"]["raw"] / cfg["archivo"]
        df = pd.read_csv(ruta, dtype=str, na_values=config["valores_nulos"], **cfg["lectura"])
        df = df.rename(columns=cfg["columnas"])[COLUMNAS_ESTANDAR].copy()
        df["fuente"] = fuente
        df["id_sucursal"] = cfg["id_sucursal"]
        partes.append(df)
    return pd.concat(partes, ignore_index=True)

In [9]:
ventas_raw = ejecutar_paso("Extracción de ventas", extraer_ventas, CONFIG)
ventas_raw.head() if ventas_raw is not None else None

,id_venta,fecha,cod_producto,id_cliente,vendedor,cantidad,precio_unitario,descuento,fuente,id_sucursal
0,AMB-00001,2025-01-01,TA-0024,C0216,maría josé torres,1,9.09,0.0,ambato,1
1,AMB-00002,2025-01-01,TA-0025,C0137,sofía paredes,1,91.73,0.15,ambato,1
2,AMB-00003,2025-01-02,TA-0031,C0177,SOFIA PAREDES,2,57.61,0.0,ambato,1
3,AMB-00004,2025-01-02,TA-0024,C0164,MARIA JOSE TORRES,1,9.3,0.0,ambato,1
4,AMB-00005,2025-01-04,TA-0001,C0017,sofía paredes,3,857.95,0.0,ambato,1


**Verificación del Ejercicio 2**

In [10]:
verificar("Se extrajeron 1914 filas en total", lambda: len(ventas_raw) == 1914)
verificar("Las columnas son el esquema estándar + fuente + id_sucursal",
          lambda: list(ventas_raw.columns) == COLUMNAS_ESTANDAR + ["fuente", "id_sucursal"])
verificar("Filas por sucursal: ambato 734, latacunga 649, pillaro 531",
          lambda: ventas_raw["fuente"].value_counts().to_dict() == {"ambato": 734, "latacunga": 649, "pillaro": 531})
verificar("Las tildes de Latacunga se leyeron correctamente",
          lambda: ventas_raw.loc[ventas_raw["fuente"] == "latacunga", "vendedor"].str.contains("Chávez", na=False).any())

[CORRECTO] Se extrajeron 1914 filas en total
[CORRECTO] Las columnas son el esquema estándar + fuente + id_sucursal
[CORRECTO] Filas por sucursal: ambato 734, latacunga 649, pillaro 531
[CORRECTO] Las tildes de Latacunga se leyeron correctamente


True

### 3.2. Catálogo: JSON con metadatos y estructura anidada

Se observa la estructura del archivo antes de extraer:

In [11]:
with open(CONFIG["rutas"]["raw"] / CONFIG["catalogo"], "r", encoding="utf-8") as f:
    catalogo_json = json.load(f)

print("Claves de primer nivel:", list(catalogo_json.keys()))
print("Metadatos:", json.dumps(catalogo_json["metadata"], indent=2, ensure_ascii=False))
print("Claves dentro de data:", list(catalogo_json["data"].keys()))
print("Claves de una categoría:", list(catalogo_json["data"]["categorias"][0].keys()))
print("Un producto:", json.dumps(catalogo_json["data"]["categorias"][0]["productos"][0], indent=2, ensure_ascii=False))

Claves de primer nivel: ['metadata', 'data']
Metadatos: {
  "sistema": "ERP TecnoAndes",
  "modulo": "catalogo",
  "version_esquema": "2.1",
  "fecha_exportacion": "2026-07-01T08:15:00",
  "moneda": "USD",
  "total_categorias": 6
}
Claves dentro de data: ['categorias']
Claves de una categoría: ['id_categoria', 'nombre_categoria', 'total_productos', 'productos']
Un producto: {
  "sku": "TA-0001",
  "nombre": "Laptop TP-Link K92",
  "marca": "TP-Link",
  "precio_lista": 876.13,
  "estado": "activo",
  "especificaciones": {
    "garantia_meses": 6,
    "peso_kg": 2.08
  }
}


La jerarquía es: `metadata` + `data → categorias → productos → especificaciones`. Los productos están **dentro** de cada categoría, por lo que hay que expandirlos con `record_path` y conservar los datos de la categoría con `meta`.

#### Ejercicio 3. Extraer el catálogo de productos

Implemente `extraer_catalogo(config)` que:

1. Cargue el JSON con `json.load`.
2. Verifique que la cantidad de categorías recibidas coincide con `metadata["total_categorias"]`; si no coincide, registre una advertencia con `logger.warning`.
3. Use `pd.json_normalize` sobre `data["categorias"]` con `record_path="productos"` y `meta=["id_categoria", "nombre_categoria"]`.
4. Devuelva el DataFrame resultante.

In [12]:
def extraer_catalogo(config: dict) -> pd.DataFrame:
    with open(config["rutas"]["raw"] / config["catalogo"], encoding="utf-8") as f:
        datos = json.load(f)
    categorias = datos["data"]["categorias"]
    if len(categorias) != datos["metadata"]["total_categorias"]:
        logger.warning("Cantidad de categorías distinta de los metadatos")
    return pd.json_normalize(categorias, record_path="productos",
                             meta=["id_categoria", "nombre_categoria"])

In [13]:
catalogo_raw = ejecutar_paso("Extracción del catálogo", extraer_catalogo, CONFIG)
catalogo_raw.head() if catalogo_raw is not None else None

,sku,nombre,marca,precio_lista,estado,especificaciones.garantia_meses,especificaciones.peso_kg,id_categoria,nombre_categoria
0,TA-0001,Laptop TP-Link K92,TP-Link,876.13,activo,6,2.08,1,Computación
1,TA-0002,PC de escritorio Sony M80,Sony,952.37,activo,24,8.33,1,Computación
2,TA-0003,PC de escritorio Samsung M20,SAMSUNG,843.96,activo,6,6.86,1,Computación
3,TA-0004,Laptop Samsung D60,Samsung,955.73,activo,12,1.69,1,Computación
4,TA-0005,Tablet Kingston K91,Kingston,479.91,activo,6,0.54,1,Computación


**Verificación del Ejercicio 3**

In [14]:
verificar("Se extrajeron 61 filas de productos", lambda: len(catalogo_raw) == 61)
verificar("Las especificaciones se aplanaron en columnas",
          lambda: {"especificaciones.garantia_meses", "especificaciones.peso_kg"} <= set(catalogo_raw.columns))
verificar("Cada producto conserva su categoría",
          lambda: catalogo_raw[["id_categoria", "nombre_categoria"]].notna().all().all())

[CORRECTO] Se extrajeron 61 filas de productos
[CORRECTO] Las especificaciones se aplanaron en columnas
[CORRECTO] Cada producto conserva su categoría


True

**Pregunta.** El catálogo tiene 60 productos, pero se extrajeron 61 filas. ¿Qué explica la diferencia? Escriba el código que lo demuestre en la celda siguiente.

In [15]:
duplicados_catalogo = catalogo_raw.loc[catalogo_raw["sku"].duplicated(keep=False)]
print("Filas:", len(catalogo_raw), "| SKU únicos:", catalogo_raw["sku"].nunique())
duplicados_catalogo

Filas: 61 | SKU únicos: 60


,sku,nombre,marca,precio_lista,estado,especificaciones.garantia_meses,especificaciones.peso_kg,id_categoria,nombre_categoria
4,TA-0005,Tablet Kingston K91,Kingston,479.91,activo,6,0.54,1,Computación
20,TA-0005,Tablet Kingston K91,Kingston,479.91,activo,6,0.54,2,Periféricos


**Respuesta:** Hay un SKU repetido en el catálogo: se extraen 61 filas, pero solo
60 productos distintos. La tabla anterior muestra ambas apariciones. R9 indica
conservar la primera; no corresponde eliminar productos por su nombre ni promediar sus valores.

### 3.3. Clientes: varios archivos JSON por lotes

El CRM exporta los clientes en tres lotes. Cada archivo tiene sus propios metadatos:

In [16]:
with open(CONFIG["rutas"]["raw"] / "crm" / "clientes_lote_1.json", "r", encoding="utf-8") as f:
    lote_ejemplo = json.load(f)

print("Claves:", list(lote_ejemplo.keys()))
print("Metadatos:", lote_ejemplo["metadata"])
print("Un registro:", json.dumps(lote_ejemplo["registros"][0], indent=2, ensure_ascii=False))

Claves: ['metadata', 'registros']
Metadatos: {'sistema': 'CRM TecnoAndes', 'lote': 1, 'total_lotes': 3, 'registros_lote': 100, 'fecha_generacion': '2026-07-01T07:00:00'}
Un registro: {
  "id_cliente": "C0014",
  "nombre_completo": "Raúl Sánchez Ortiz",
  "contacto": {
    "email": "raul.sanchez14@correo.ec",
    "telefono": "0985201538"
  },
  "ubicacion": {
    "ciudad": "Riobamba",
    "provincia": "Chimborazo"
  },
  "segmento": "Empresa",
  "fecha_registro": "2024-10-09",
  "fecha_actualizacion": "2025-01-01T10:00:00"
}


#### Ejercicio 4. Extraer y unir los lotes de clientes

Implemente `extraer_clientes(config)` que:

1. Recorra, **en orden**, los archivos que coinciden con `config["clientes_patron"]` (use `sorted(ruta_raw.glob(...))`).
2. Para cada archivo: cargue el JSON, verifique que `len(registros)` coincida con `metadata["registros_lote"]` (si no coincide, `logger.warning`), normalice `registros` con `pd.json_normalize` y agregue una columna `lote` con `metadata["lote"]`.
3. Concatene todos los lotes con `ignore_index=True`.

In [17]:
def extraer_clientes(config: dict) -> pd.DataFrame:
    partes = []
    rutas = sorted(config["rutas"]["raw"].glob(config["clientes_patron"]))
    if not rutas:
        raise FileNotFoundError("No se encontraron lotes del CRM")
    for ruta in rutas:
        with open(ruta, encoding="utf-8") as f:
            datos = json.load(f)
        registros, metadata = datos["registros"], datos["metadata"]
        if len(registros) != metadata["registros_lote"]:
            logger.warning("%s: cantidad distinta de los metadatos", ruta.name)
        df = pd.json_normalize(registros)
        df["lote"] = metadata["lote"]
        partes.append(df)
    return pd.concat(partes, ignore_index=True)

In [18]:
clientes_raw = ejecutar_paso("Extracción de clientes", extraer_clientes, CONFIG)
clientes_raw.head() if clientes_raw is not None else None

,id_cliente,nombre_completo,segmento,fecha_registro,fecha_actualizacion,contacto.email,contacto.telefono,ubicacion.ciudad,ubicacion.provincia,lote
0,C0014,Raúl Sánchez Ortiz,Empresa,2024-10-09,2025-01-01T10:00:00,raul.sanchez14@correo.ec,0985201538,Riobamba,Chimborazo,1
1,C0048,Pedro Yánez Núñez,Empresa,2024-07-15,2025-05-02T10:00:00,pedro.yanez48@correo.ec,0985895725,Riobamba,Chimborazo,1
2,C0061,Diego Vargas Vargas,Empresa,2024-05-15,2024-09-26T10:00:00,diego.vargas61@correo.ec,0991664672,Salcedo,Cotopaxi,1
3,C0093,Raúl Ortiz Guamán,Hogar,2024-10-17,2025-03-12T10:00:00,raul.ortiz93@correo.ec,0990456596,Ambato,Tungurahua,1
4,C0060,Paola Núñez Pérez,Estudiante,2024-09-27,2025-07-15T10:00:00,paola.nunez60@correo.ec,0986968099,Riobamba,Chimborazo,1


**Verificación del Ejercicio 4**

In [19]:
verificar("Se extrajeron 320 registros de clientes", lambda: len(clientes_raw) == 320)
verificar("Existen los tres lotes", lambda: sorted(clientes_raw["lote"].unique()) == [1, 2, 3])
verificar("Los objetos anidados se aplanaron (contacto.email, ubicacion.ciudad)",
          lambda: {"contacto.email", "ubicacion.ciudad"} <= set(clientes_raw.columns))
verificar("Hay clientes repetidos entre lotes (300 identificadores únicos)",
          lambda: clientes_raw["id_cliente"].nunique() == 300)

[CORRECTO] Se extrajeron 320 registros de clientes
[CORRECTO] Existen los tres lotes
[CORRECTO] Los objetos anidados se aplanaron (contacto.email, ubicacion.ciudad)
[CORRECTO] Hay clientes repetidos entre lotes (300 identificadores únicos)


True

### 3.4. Vendedores: Excel con encabezado desplazado y fila de totales

Se observa la hoja tal como está, sin interpretar encabezados:

In [20]:
pd.read_excel(CONFIG["rutas"]["raw"] / CONFIG["vendedores"]["archivo"],
              sheet_name=CONFIG["vendedores"]["hoja"], header=None)

,0,1,2,3,4
0,TecnoAndes Cía. Ltda. - Talento Humano,NaN,NaN,NaN,NaN
1,Corte: 30/06/2026,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,Nombre,Sucursal,Fecha de ingreso,Comisión,Estado
4,María José Torres,Ambato,2021-03-15 00:00:00,5%,Activo
5,Andrés Ruiz,Ambato,2022-08-01 00:00:00,4%,Activo
6,Sofía Paredes,AMBATO,2024-01-10 00:00:00,3%,Activo
7,José Luis Vega,latacunga,2020-11-20 00:00:00,5%,Activo
8,Lucía Mena,Latacunga,2023-05-02 00:00:00,4%,Activo
9,Diego Chávez,latacunga,2024-09-16 00:00:00,3%,Activo


#### Ejercicio 5. Extraer la nómina de vendedores

Implemente `extraer_vendedores(config)` que:

1. Lea la hoja indicada en `config["vendedores"]["hoja"]` con el encabezado en `config["vendedores"]["fila_encabezado"]`.
2. Elimine las filas completamente vacías.
3. Elimine la fila de totales (la que tiene `"TOTAL"` en la columna `Nombre`).

In [21]:
def extraer_vendedores(config: dict) -> pd.DataFrame:
    cfg = config["vendedores"]
    df = pd.read_excel(config["rutas"]["raw"] / cfg["archivo"],
                       sheet_name=cfg["hoja"], header=cfg["fila_encabezado"])
    df = df.dropna(how="all")
    return df.loc[df["Nombre"].astype("string").str.strip().str.upper().ne("TOTAL")].reset_index(drop=True)

In [22]:
vendedores_raw = ejecutar_paso("Extracción de vendedores", extraer_vendedores, CONFIG)
vendedores_raw

,Nombre,Sucursal,Fecha de ingreso,Comisión,Estado
0,María José Torres,Ambato,2021-03-15,5%,Activo
1,Andrés Ruiz,Ambato,2022-08-01,4%,Activo
2,Sofía Paredes,AMBATO,2024-01-10,3%,Activo
3,José Luis Vega,latacunga,2020-11-20,5%,Activo
4,Lucía Mena,Latacunga,2023-05-02,4%,Activo
5,Diego Chávez,latacunga,2024-09-16,3%,Activo
6,Gabriela Ortiz,Pillaro,2022-02-07,4%,Activo
7,Patricio Yánez,Píllaro,2023-10-30,3%,Activo
8,Carlos Núñez,Píllaro,2025-02-03,3%,Activo
9,Ramiro Paz,Ambato,2019-06-01,4%,Inactivo


**Verificación del Ejercicio 5**

In [23]:
verificar("Se extrajeron 10 vendedores", lambda: len(vendedores_raw) == 10)
verificar("Las columnas son las de la tabla",
          lambda: list(vendedores_raw.columns) == ["Nombre", "Sucursal", "Fecha de ingreso", "Comisión", "Estado"])
verificar("No queda la fila TOTAL", lambda: "TOTAL" not in vendedores_raw["Nombre"].tolist())

[CORRECTO] Se extrajeron 10 vendedores
[CORRECTO] Las columnas son las de la tabla
[CORRECTO] No queda la fila TOTAL


True

### 3.5. Inventario: base de datos SQLite

Una base de datos se consulta con SQL. Primero se listan las tablas disponibles:

In [24]:
with closing(sqlite3.connect(CONFIG["rutas"]["raw"] / CONFIG["inventario"])) as con:
    tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con)
    display(tablas)
    for t in tablas["name"]:
        print(f"--- {t} ---")
        display(pd.read_sql(f"SELECT * FROM {t} LIMIT 3", con))

,name
0,sucursales
1,costos
2,stock


--- sucursales ---


,id_sucursal,nombre,ciudad,provincia,fecha_apertura
0,1,TecnoAndes Ambato,Ambato,Tungurahua,2015-04-01
1,2,TecnoAndes Latacunga,Latacunga,Cotopaxi,2018-09-15
2,3,TecnoAndes Píllaro,Píllaro,Tungurahua,2021-02-01


--- costos ---


,product_id,costo_unitario,actualizado
0,1,662.18,2026-06-30
1,2,631.33,2026-06-30
2,3,483.21,2026-06-30


--- stock ---


,id_sucursal,product_id,cantidad,fecha_corte
0,1,1,11,2026-05-31
1,1,2,53,2026-05-31
2,1,3,17,2026-05-31


#### Ejercicio 6. Extraer el inventario con consultas SQL

Implemente `extraer_inventario(config)` que devuelva un **diccionario** con tres DataFrames, usando una consulta SQL para cada uno:

| Clave | Tabla | Columnas resultantes (use `AS` para renombrar) |
|---|---|---|
| `"sucursales"` | `sucursales` | `id_sucursal, sucursal, ciudad_sucursal, provincia_sucursal` |
| `"costos"` | `costos` | `id_producto, costo_unitario` |
| `"stock"` | `stock` | `id_sucursal, id_producto, stock_sucursal, fecha_corte` |

Para `stock`, extraiga **solo el corte más reciente** (regla R10) filtrando en SQL con una subconsulta: `WHERE fecha_corte = (SELECT MAX(fecha_corte) FROM stock)`.

Use `closing(sqlite3.connect(ruta))` para que la conexión se cierre al terminar.

In [25]:
def extraer_inventario(config: dict) -> dict:
    ruta = config["rutas"]["raw"] / config["inventario"]
    consultas = {
        "sucursales": """SELECT id_sucursal, nombre AS sucursal,
                         ciudad AS ciudad_sucursal, provincia AS provincia_sucursal
                         FROM sucursales ORDER BY id_sucursal""",
        "costos": """SELECT product_id AS id_producto, costo_unitario
                      FROM costos ORDER BY product_id""",
        "stock": """SELECT id_sucursal, product_id AS id_producto,
                     cantidad AS stock_sucursal, fecha_corte FROM stock
                     WHERE fecha_corte = (SELECT MAX(fecha_corte) FROM stock)
                     ORDER BY id_sucursal, product_id""",
    }
    with closing(sqlite3.connect(ruta.resolve().as_uri() + "?mode=ro", uri=True)) as con:
        return {nombre: pd.read_sql(consulta, con) for nombre, consulta in consultas.items()}

In [26]:
inventario_raw = ejecutar_paso("Extracción del inventario", extraer_inventario, CONFIG)
if inventario_raw:
    for nombre, df in inventario_raw.items():
        print(nombre, df.shape)

sucursales (4, 4)
costos (60, 2)
stock (180, 4)


**Verificación del Ejercicio 6**

In [27]:
verificar("Se extrajeron 4 sucursales", lambda: len(inventario_raw["sucursales"]) == 4)
verificar("Se extrajeron 60 costos", lambda: len(inventario_raw["costos"]) == 60)
verificar("Stock: solo el corte 2026-06-30 (180 filas)",
          lambda: len(inventario_raw["stock"]) == 180 and set(inventario_raw["stock"]["fecha_corte"]) == {"2026-06-30"})
verificar("Columnas de stock renombradas",
          lambda: list(inventario_raw["stock"].columns) == ["id_sucursal", "id_producto", "stock_sucursal", "fecha_corte"])

[CORRECTO] Se extrajeron 4 sucursales
[CORRECTO] Se extrajeron 60 costos
[CORRECTO] Stock: solo el corte 2026-06-30 (180 filas)
[CORRECTO] Columnas de stock renombradas


True

---
## 4. Diagnóstico de calidad

Antes de transformar se **documentan los problemas**. Cada decisión posterior debe responder a un problema identificado aquí y a una regla de negocio.

In [28]:
perfilar(ventas_raw, "Ventas")

Ventas: 1914 filas x 10 columnas | filas duplicadas: 34


,tipo,nulos,unicos,ejemplo
id_venta,str,0,1880,AMB-00001
fecha,str,0,1304,2025-01-01
cod_producto,str,0,184,TA-0024
id_cliente,str,16,561,C0216
vendedor,str,5,43,maría josé torres
cantidad,str,0,12,1
precio_unitario,str,0,1822,9.09
descuento,str,0,12,0.0
fuente,str,0,3,ambato
id_sucursal,int64,0,3,1


In [29]:
print("Filas completamente duplicadas por fuente:")
print(ventas_raw[ventas_raw.duplicated(keep="first")]["fuente"].value_counts())

print("\nValores distintos del vendedor 'María José Torres':")
print(ventas_raw.loc[ventas_raw["vendedor"].str.contains("jos", case=False, na=False) &
                     ventas_raw["vendedor"].str.contains("torres", case=False, na=False),
                     "vendedor"].value_counts())

Filas completamente duplicadas por fuente:
fuente
ambato       14
pillaro      11
latacunga     9
Name: count, dtype: int64

Valores distintos del vendedor 'María José Torres':
vendedor
María José Torres       146
MARÍA JOSÉ TORRES        38
Maria Jose Torres        36
MARIA JOSE TORRES        30
  maría josé torres      16
Name: count, dtype: int64


In [30]:
print("Ejemplos de códigos de producto por fuente:")
print(ventas_raw.groupby("fuente")["cod_producto"].first())
print("\nEjemplos de precios y descuentos por fuente:")
print(ventas_raw.groupby("fuente")[["precio_unitario", "descuento"]].first())

Ejemplos de códigos de producto por fuente:
fuente
ambato       TA-0024
latacunga     ta0060
pillaro           33
Name: cod_producto, dtype: str

Ejemplos de precios y descuentos por fuente:
          precio_unitario descuento
fuente                             
ambato               9.09       0.0
latacunga          139,51       0,0
pillaro            $85,42       10%


In [31]:
perfilar(catalogo_raw, "Catálogo")

Catálogo: 61 filas x 9 columnas | filas duplicadas: 0


,tipo,nulos,unicos,ejemplo
sku,str,0,60,TA-0001
nombre,str,0,60,Laptop TP-Link K92
marca,str,0,17,TP-Link
precio_lista,object,0,60,876.13
estado,str,0,2,activo
especificaciones.garantia_meses,int64,0,3,6
especificaciones.peso_kg,float64,4,46,2.08
id_categoria,object,0,6,1
nombre_categoria,str,0,6,Computación


In [32]:
print("Marcas escritas de forma distinta:")
print(sorted(catalogo_raw["marca"].unique()))
print("\nTipos de dato encontrados en precio_lista:")
print(catalogo_raw["precio_lista"].map(type).value_counts())

Marcas escritas de forma distinta:
['Dell', 'EPSON', 'Epson', 'HP', 'Kingston', 'LENOVO', 'Lenovo', 'SAMSUNG', 'Samsung', 'Sony', 'TP-LINK', 'TP-Link', 'XIAOMI', 'Xiaomi', 'epson', 'lenovo', 'sony']

Tipos de dato encontrados en precio_lista:
precio_lista
<class 'float'>    50
<class 'str'>      11
Name: count, dtype: int64


In [33]:
perfilar(clientes_raw, "Clientes")

Clientes: 320 filas x 10 columnas | filas duplicadas: 0


,tipo,nulos,unicos,ejemplo
id_cliente,str,0,300,C0014
nombre_completo,str,0,294,Raúl Sánchez Ortiz
segmento,str,10,3,Empresa
fecha_registro,str,0,229,2024-10-09
fecha_actualizacion,str,0,251,2025-01-01T10:00:00
contacto.email,str,0,300,raul.sanchez14@correo.ec
contacto.telefono,str,0,300,0985201538
ubicacion.ciudad,str,0,14,Riobamba
ubicacion.provincia,str,0,3,Chimborazo
lote,int64,0,3,1


In [34]:
print("Valores de ciudad:")
print(clientes_raw["ubicacion.ciudad"].value_counts())

Valores de ciudad:
ubicacion.ciudad
Latacunga      63
Riobamba       51
Ambato         51
Pelileo        48
Salcedo        47
Píllaro        44
SALCEDO         3
 píllaro        3
PELILEO         2
 pelileo        2
 salcedo        2
PÍLLARO         2
 riobamba       1
 latacunga      1
Name: count, dtype: int64


In [35]:
print("Stocks negativos en el último corte:", (inventario_raw["stock"]["stock_sucursal"] < 0).sum())
print("Productos sin costo:", inventario_raw["costos"]["costo_unitario"].isna().sum())

Stocks negativos en el último corte: 6
Productos sin costo: 3


#### Ejercicio 7. Registro de problemas detectados

Con base en los perfiles anteriores, complete un DataFrame `problemas` con **al menos 10 problemas** de calidad, uno por fila, con las columnas `fuente`, `problema`, `evidencia` y `regla` (código de la regla de negocio que lo resuelve, o `"Estandarización"` si se resuelve al convertir formatos).

In [36]:
def diagnosticar_problemas(v, cat, cli, vend, inv, config):
    cantidades = pd.to_numeric(v["cantidad"], errors="coerce")
    productos = pd.to_numeric(v["cod_producto"].str.extract(r"(\d+)", expand=False))
    ids_catalogo = pd.to_numeric(cat["sku"].str.extract(r"(\d+)", expand=False))
    ids_cliente = pd.to_numeric(v["id_cliente"].str.extract(r"(\d+)", expand=False))
    crm_ids = pd.to_numeric(cli["id_cliente"].str.extract(r"(\d+)", expand=False))
    filas = [
        ("Ventas", "Filas duplicadas", f"{v.duplicated().sum()} filas completas repetidas", "R1"),
        ("Ventas", "Cantidad no válida", f"{(cantidades.isna() | cantidades.le(0)).sum()} filas crudas", "R2"),
        ("Ventas", "Producto fuera del catálogo", f"{(~productos.isin(ids_catalogo)).sum()} filas crudas", "R3"),
        ("Ventas", "Cliente ausente", f"{v['id_cliente'].isna().sum()} filas crudas", "R4"),
        ("Ventas", "Cliente ajeno al CRM", f"{(ids_cliente.notna() & ~ids_cliente.isin(crm_ids)).sum()} filas crudas", "R5"),
        ("Ventas", "Vendedor ausente", f"{v['vendedor'].isna().sum()} filas crudas", "R6"),
        ("Ventas", "Cantidades atípicas", f"{cantidades.gt(config['umbral_cantidad_atipica']).sum()} filas crudas", "R7"),
        ("Ventas", "Formatos distintos", "Tres formatos de fecha; punto/coma decimal; descuentos en proporción o %", "Estandarización"),
        ("Ventas", "Nombres con variantes", f"{v['vendedor'].nunique()} nombres originales para 9 vendedores utilizados", "Estandarización"),
        ("Catálogo", "SKU repetido", f"{cat['sku'].duplicated().sum()} aparición adicional", "R9"),
        ("Catálogo", "Variantes de marca", f"{cat['marca'].nunique()} marcas antes de normalizar", "Estandarización"),
        ("Catálogo", "Precios con tipos mixtos", str(cat['precio_lista'].map(lambda x: type(x).__name__).value_counts().to_dict()), "Estandarización"),
        ("Clientes", "Versiones entre lotes", f"{cli['id_cliente'].duplicated().sum()} registros adicionales", "R8"),
        ("Clientes", "Ciudad inconsistente", f"{cli['ubicacion.ciudad'].nunique()} grafías originales", "Estandarización"),
        ("Clientes", "Segmentos faltantes", f"{cli['segmento'].isna().sum()} nulos antes de depurar versiones", "Estandarización"),
        ("Vendedores", "Encabezado desplazado y fila TOTAL", "Personal: encabezado en índice 3; TOTAL y una fila vacía excluidas", "Estandarización"),
        ("Vendedores", "Comisión textual", str(vend['Comisión'].unique().tolist()), "Estandarización"),
        ("Inventario", "Stocks negativos", f"{inv['stock']['stock_sucursal'].lt(0).sum()} en el último corte", "R10"),
        ("Inventario", "Costos ausentes", f"{inv['costos']['costo_unitario'].isna().sum()} productos sin costo", "R11"),
    ]
    return pd.DataFrame(filas, columns=["fuente", "problema", "evidencia", "regla"])

problemas = diagnosticar_problemas(ventas_raw, catalogo_raw, clientes_raw,
                                  vendedores_raw, inventario_raw, CONFIG)
problemas

,fuente,problema,evidencia,regla
0,Ventas,Filas duplicadas,34 filas completas repetidas,R1
1,Ventas,Cantidad no válida,16 filas crudas,R2
2,Ventas,Producto fuera del catálogo,7 filas crudas,R3
3,Ventas,Cliente ausente,16 filas crudas,R4
4,Ventas,Cliente ajeno al CRM,6 filas crudas,R5
5,Ventas,Vendedor ausente,5 filas crudas,R6
6,Ventas,Cantidades atípicas,5 filas crudas,R7
7,Ventas,Formatos distintos,Tres formatos de fecha; punto/coma decimal; de...,Estandarización
8,Ventas,Nombres con variantes,43 nombres originales para 9 vendedores utiliz...,Estandarización
9,Catálogo,SKU repetido,1 aparición adicional,R9


**Verificación del Ejercicio 7**

In [37]:
verificar("Se documentaron al menos 10 problemas", lambda: len(problemas) >= 10)
verificar("Se cubren las cinco fuentes",
          lambda: problemas["fuente"].nunique() >= 5)

[CORRECTO] Se documentaron al menos 10 problemas
[CORRECTO] Se cubren las cinco fuentes


True

---
## 5. Estandarización

En esta etapa cada fuente se lleva a un formato consistente: tipos de dato correctos, claves de unión comparables y reglas de negocio aplicadas.

### 5.1. Funciones de normalización de claves

La unión entre fuentes solo funciona si las claves son **idénticas**. `"TA-0001"`, `"ta0001"` y `"1"` representan el mismo producto, y `"  maría josé torres "` y `"MARIA JOSE TORRES"` el mismo vendedor, pero para Pandas son valores distintos.

#### Ejercicio 8. Funciones de normalización

Implemente tres funciones que reciben una `pd.Series` y devuelven una `pd.Series`:

- `normalizar_texto(serie)`: convierte a texto, elimina espacios al inicio y al final, reduce espacios múltiples a uno, convierte a mayúsculas y elimina tildes. Los nulos deben seguir siendo nulos. Para quitar tildes: `unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")`.
- `extraer_numero_id(serie)`: extrae la parte numérica de un identificador (`"TA-0001"` → `1`, `"CLI-12"` → `12`) y la devuelve como entero nullable (`"Int64"`). Use `str.extract(r"(\d+)", expand=False)`.
- `a_numero(serie, decimal=".")`: convierte texto a número eliminando `$`, `%` y espacios. Si `decimal == ","`, primero elimina los puntos de miles y luego cambia la coma por punto. Use `pd.to_numeric(..., errors="coerce")` y devuelva siempre decimales (`.astype("float64")`).

In [38]:
def normalizar_texto(serie: pd.Series) -> pd.Series:
    texto = serie.astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.upper()
    return texto.map(lambda t: unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
                     if pd.notna(t) else pd.NA).astype("string")


def extraer_numero_id(serie: pd.Series) -> pd.Series:
    return pd.to_numeric(serie.astype("string").str.extract(r"(\d+)", expand=False),
                         errors="coerce").astype("Int64")


def a_numero(serie: pd.Series, decimal: str = ".") -> pd.Series:
    texto = serie.astype("string").str.replace(r"[$%\s]", "", regex=True)
    if decimal == ",":
        texto = texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(texto, errors="coerce").astype("float64")

**Verificación del Ejercicio 8**

In [39]:
prueba_txt = pd.Series(["  maría josé   torres ", "MARIA JOSE TORRES", "María José Torres", None])
verificar("normalizar_texto unifica las variantes del nombre",
          lambda: normalizar_texto(prueba_txt)[:3].nunique() == 1
          and normalizar_texto(prueba_txt)[0] == "MARIA JOSE TORRES")
verificar("normalizar_texto conserva los nulos", lambda: pd.isna(normalizar_texto(prueba_txt)[3]))
verificar("extraer_numero_id: TA-0001, ta0001, 1, CLI-1 -> 1",
          lambda: extraer_numero_id(pd.Series(["TA-0001", "ta0001", "1", "CLI-1"])).tolist() == [1, 1, 1, 1])
verificar("extraer_numero_id devuelve Int64 y conserva nulos",
          lambda: str(extraer_numero_id(pd.Series(["C0005", None])).dtype) == "Int64"
          and pd.isna(extraer_numero_id(pd.Series(["C0005", None]))[1]))
verificar("a_numero con coma decimal: '$1.250,50' -> 1250.5",
          lambda: a_numero(pd.Series(["$1.250,50"]), ",")[0] == 1250.5)
verificar("a_numero con punto decimal: '9.09' -> 9.09 y '15%' -> 15",
          lambda: a_numero(pd.Series(["9.09", "15%"])).tolist() == [9.09, 15.0])

[CORRECTO] normalizar_texto unifica las variantes del nombre
[CORRECTO] normalizar_texto conserva los nulos
[CORRECTO] extraer_numero_id: TA-0001, ta0001, 1, CLI-1 -> 1
[CORRECTO] extraer_numero_id devuelve Int64 y conserva nulos
[CORRECTO] a_numero con coma decimal: '$1.250,50' -> 1250.5
[CORRECTO] a_numero con punto decimal: '9.09' -> 9.09 y '15%' -> 15


True

### 5.2. Estandarización de ventas

#### Ejercicio 9. Estandarizar las ventas

Implemente `estandarizar_ventas(df, config)` que devuelva una tupla `(ventas, rechazados)`:

1. **R1:** elimine filas duplicadas y registre cuántas se eliminaron.
2. Por cada fuente (use la configuración de esa sucursal):
   - Convierta `fecha` con `pd.to_datetime(..., format=cfg["formato_fecha"])` y quite la hora con `.dt.normalize()`.
   - Convierta `precio_unitario` con `a_numero` y el decimal de la fuente.
   - Convierta `descuento` con `a_numero`; si el texto original tenía `%`, divida entre 100.
   - Si la fuente tiene `prefijo_id`, construya `id_venta` como prefijo + número con 5 dígitos (`str.zfill(5)`).
3. Cree `id_producto` con `extraer_numero_id(cod_producto)`.
4. Construya `id_cliente` con el formato `C0001`. Los vacíos se reemplazan por `config["cliente_consumidor_final"]` (**R4**).
5. Cree `vendedor_clave = normalizar_texto(vendedor)`; los vacíos se reemplazan por `"SIN ASIGNAR"` (**R6**).
6. Convierta `cantidad` a `"Int64"`. **R2:** separe en `rechazados` las filas con cantidad nula o menor o igual a cero, con una columna `motivo`.

In [40]:
def estandarizar_ventas(df: pd.DataFrame, config: dict):
    duplicados = int(df.duplicated().sum())
    df = df.drop_duplicates().copy()
    logger.info("R1: %s duplicados eliminados", duplicados)
    partes = []
    for fuente, cfg in config["fuentes_ventas"].items():
        parte = df.loc[df["fuente"].eq(fuente)].copy()
        parte["fecha"] = pd.to_datetime(parte["fecha"], format=cfg["formato_fecha"]).dt.normalize()
        parte["precio_unitario"] = a_numero(parte["precio_unitario"], cfg["decimal"])
        es_porcentaje = parte["descuento"].str.contains("%", regex=False, na=False)
        parte["descuento"] = a_numero(parte["descuento"], cfg["decimal"])
        parte.loc[es_porcentaje, "descuento"] /= 100
        if cfg["prefijo_id"]:
            parte["id_venta"] = cfg["prefijo_id"] + parte["id_venta"].str.strip().str.zfill(5)
        partes.append(parte)
    df = pd.concat(partes, ignore_index=True)
    df["id_producto"] = extraer_numero_id(df["cod_producto"])
    cliente_num = extraer_numero_id(df["id_cliente"])
    df["id_cliente"] = ("C" + cliente_num.astype("string").str.zfill(4)).fillna(
        config["cliente_consumidor_final"])
    df["vendedor_clave"] = normalizar_texto(df["vendedor"]).replace("", pd.NA).fillna("SIN ASIGNAR")
    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").astype("Int64")
    invalida = df["cantidad"].isna() | df["cantidad"].le(0)
    rechazados = df.loc[invalida].copy()
    rechazados["motivo"] = "R2: cantidad nula o menor o igual a cero"
    logger.info("R2: %s líneas rechazadas por cantidad", len(rechazados))
    return df.loc[~invalida].reset_index(drop=True), rechazados.reset_index(drop=True)

In [41]:
resultado = ejecutar_paso("Estandarización de ventas", estandarizar_ventas, ventas_raw, CONFIG)
ventas, rechazados_cantidad = resultado if resultado else (None, None)
ventas.head() if ventas is not None else None

INFO | R1: 34 duplicados eliminados


INFO | R2: 16 líneas rechazadas por cantidad


,id_venta,fecha,cod_producto,id_cliente,vendedor,cantidad,precio_unitario,descuento,fuente,id_sucursal,id_producto,vendedor_clave
0,AMB-00001,2025-01-01,TA-0024,C0216,maría josé torres,1,9.09,0.00,ambato,1,24,MARIA JOSE TORRES
1,AMB-00002,2025-01-01,TA-0025,C0137,sofía paredes,1,91.73,0.15,ambato,1,25,SOFIA PAREDES
2,AMB-00003,2025-01-02,TA-0031,C0177,SOFIA PAREDES,2,57.61,0.00,ambato,1,31,SOFIA PAREDES
3,AMB-00004,2025-01-02,TA-0024,C0164,MARIA JOSE TORRES,1,9.30,0.00,ambato,1,24,MARIA JOSE TORRES
4,AMB-00005,2025-01-04,TA-0001,C0017,sofía paredes,3,857.95,0.00,ambato,1,1,SOFIA PAREDES


**Verificación del Ejercicio 9**

In [42]:
verificar("Quedan 1864 ventas válidas", lambda: len(ventas) == 1864)
verificar("Se rechazaron 16 filas por cantidad", lambda: len(rechazados_cantidad) == 16)
verificar("id_venta es único", lambda: ventas["id_venta"].is_unique)
verificar("id_venta de Latacunga con formato LAT-00001",
          lambda: ventas.loc[ventas["fuente"] == "latacunga", "id_venta"].str.match(r"^LAT-\d{5}$").all())
verificar("fecha es datetime y sin nulos",
          lambda: pd.api.types.is_datetime64_any_dtype(ventas["fecha"]) and ventas["fecha"].notna().all())
verificar("precio_unitario numérico y sin nulos",
          lambda: pd.api.types.is_numeric_dtype(ventas["precio_unitario"]) and ventas["precio_unitario"].notna().all())
verificar("descuento entre 0 y 0.15", lambda: ventas["descuento"].between(0, 0.15).all())
verificar("id_cliente con formato C0000", lambda: ventas["id_cliente"].str.match(r"^C\d{4}$").all())
verificar("15 ventas asignadas a consumidor final", lambda: (ventas["id_cliente"] == "C0000").sum() == 15)
verificar("vendedor_clave sin nulos y con 10 valores distintos (9 vendedores + SIN ASIGNAR)",
          lambda: ventas["vendedor_clave"].notna().all() and ventas["vendedor_clave"].nunique() == 10)

[CORRECTO] Quedan 1864 ventas válidas
[CORRECTO] Se rechazaron 16 filas por cantidad
[CORRECTO] id_venta es único
[CORRECTO] id_venta de Latacunga con formato LAT-00001
[CORRECTO] fecha es datetime y sin nulos
[CORRECTO] precio_unitario numérico y sin nulos
[CORRECTO] descuento entre 0 y 0.15
[CORRECTO] id_cliente con formato C0000
[CORRECTO] 15 ventas asignadas a consumidor final
[CORRECTO] vendedor_clave sin nulos y con 10 valores distintos (9 vendedores + SIN ASIGNAR)


True

### 5.3. Estandarización de las fuentes maestras

#### Ejercicio 10. Estandarizar catálogo, clientes y vendedores

Implemente las tres funciones siguientes.

**`estandarizar_catalogo(df)`**
- Renombre: `nombre → producto`, `nombre_categoria → categoria`, `estado → estado_producto`, `especificaciones.garantia_meses → garantia_meses`, `especificaciones.peso_kg → peso_kg`.
- Cree `id_producto` a partir de `sku`; convierta `precio_lista` a número; deje `marca` en mayúsculas y sin espacios sobrantes.
- **R9:** elimine los `id_producto` repetidos conservando la primera aparición.
- Devuelva las columnas: `id_producto, sku, producto, marca, categoria, estado_producto, precio_lista, garantia_meses, peso_kg`.

**`estandarizar_clientes(df)`**
- Renombre: `nombre_completo → cliente`, `contacto.email → email`, `contacto.telefono → telefono`, `ubicacion.ciudad → ciudad_cliente`, `ubicacion.provincia → provincia_cliente`.
- Convierta `fecha_actualizacion` y `fecha_registro` a fecha.
- **R8:** ordene por `id_cliente` y `fecha_actualizacion` descendente, y conserve el primero de cada cliente.
- `ciudad_cliente` sin espacios sobrantes y en formato título; `email` en minúsculas; `segmento` nulo → `"No informado"`.
- Devuelva: `id_cliente, cliente, email, telefono, ciudad_cliente, provincia_cliente, segmento, fecha_registro`.

**`estandarizar_vendedores(df)`**
- Renombre: `Nombre → vendedor`, `Fecha de ingreso → fecha_ingreso`, `Comisión → comision_pct`, `Estado → estado_vendedor`.
- Cree `vendedor_clave` con `normalizar_texto`; convierta `comision_pct` a proporción (`"5%"` → `0.05`).
- Devuelva: `vendedor_clave, vendedor, fecha_ingreso, comision_pct, estado_vendedor`.

In [43]:
def estandarizar_catalogo(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={"nombre": "producto", "nombre_categoria": "categoria",
        "estado": "estado_producto", "especificaciones.garantia_meses": "garantia_meses",
        "especificaciones.peso_kg": "peso_kg"}).copy()
    df["id_producto"] = extraer_numero_id(df["sku"])
    df["precio_lista"] = a_numero(df["precio_lista"])
    df["marca"] = df["marca"].astype("string").str.strip().str.upper()
    df = df.drop_duplicates("id_producto", keep="first")
    return df[["id_producto", "sku", "producto", "marca", "categoria", "estado_producto",
               "precio_lista", "garantia_meses", "peso_kg"]].reset_index(drop=True)


def estandarizar_clientes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={"nombre_completo": "cliente", "contacto.email": "email",
        "contacto.telefono": "telefono", "ubicacion.ciudad": "ciudad_cliente",
        "ubicacion.provincia": "provincia_cliente"}).copy()
    for columna in ["fecha_actualizacion", "fecha_registro"]:
        df[columna] = pd.to_datetime(df[columna])
    df = df.sort_values(["id_cliente", "fecha_actualizacion"], ascending=[True, False],
                         kind="stable").drop_duplicates("id_cliente", keep="first")
    df["ciudad_cliente"] = df["ciudad_cliente"].astype("string").str.strip().str.title()
    df["email"] = df["email"].astype("string").str.strip().str.lower()
    df["segmento"] = df["segmento"].fillna("No informado")
    return df[["id_cliente", "cliente", "email", "telefono", "ciudad_cliente",
               "provincia_cliente", "segmento", "fecha_registro"]].reset_index(drop=True)


def estandarizar_vendedores(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={"Nombre": "vendedor", "Fecha de ingreso": "fecha_ingreso",
        "Comisión": "comision_pct", "Estado": "estado_vendedor"}).copy()
    df["vendedor"] = df["vendedor"].astype("string").str.strip()
    df["vendedor_clave"] = normalizar_texto(df["vendedor"])
    df["comision_pct"] = a_numero(df["comision_pct"]) / 100
    df["fecha_ingreso"] = pd.to_datetime(df["fecha_ingreso"])
    return df[["vendedor_clave", "vendedor", "fecha_ingreso", "comision_pct", "estado_vendedor"]]

In [44]:
catalogo = ejecutar_paso("Estandarización del catálogo", estandarizar_catalogo, catalogo_raw)
clientes = ejecutar_paso("Estandarización de clientes", estandarizar_clientes, clientes_raw)
vendedores = ejecutar_paso("Estandarización de vendedores", estandarizar_vendedores, vendedores_raw)
vendedores

,vendedor_clave,vendedor,fecha_ingreso,comision_pct,estado_vendedor
0,MARIA JOSE TORRES,María José Torres,2021-03-15,0.05,Activo
1,ANDRES RUIZ,Andrés Ruiz,2022-08-01,0.04,Activo
2,SOFIA PAREDES,Sofía Paredes,2024-01-10,0.03,Activo
3,JOSE LUIS VEGA,José Luis Vega,2020-11-20,0.05,Activo
4,LUCIA MENA,Lucía Mena,2023-05-02,0.04,Activo
5,DIEGO CHAVEZ,Diego Chávez,2024-09-16,0.03,Activo
6,GABRIELA ORTIZ,Gabriela Ortiz,2022-02-07,0.04,Activo
7,PATRICIO YANEZ,Patricio Yánez,2023-10-30,0.03,Activo
8,CARLOS NUNEZ,Carlos Núñez,2025-02-03,0.03,Activo
9,RAMIRO PAZ,Ramiro Paz,2019-06-01,0.04,Inactivo


**Verificación del Ejercicio 10**

In [45]:
verificar("Catálogo: 60 productos con id_producto único",
          lambda: len(catalogo) == 60 and catalogo["id_producto"].is_unique)
verificar("Catálogo: precio_lista numérico", lambda: pd.api.types.is_numeric_dtype(catalogo["precio_lista"]))
verificar("Catálogo: marcas sin variantes de mayúsculas (9 marcas)", lambda: catalogo["marca"].nunique() == 9)
verificar("Clientes: 300 únicos", lambda: len(clientes) == 300 and clientes["id_cliente"].is_unique)
verificar("Clientes: se conservó la versión más reciente de cada cliente (R8)",
          lambda: (clientes_raw.assign(_f=pd.to_datetime(clientes_raw["fecha_actualizacion"]))
                   .sort_values("_f").drop_duplicates("id_cliente", keep="last")
                   .set_index("id_cliente")["segmento"].fillna("No informado").astype(str).sort_index()
                   == clientes.set_index("id_cliente")["segmento"].astype(str).sort_index()).all())
verificar("Clientes: 6 ciudades bien escritas", lambda: clientes["ciudad_cliente"].nunique() == 6)
verificar("Clientes: sin segmentos nulos", lambda: clientes["segmento"].notna().all())
verificar("Vendedores: comisión como proporción (máximo 0.05)", lambda: vendedores["comision_pct"].max() == 0.05)
verificar("Vendedores: clave normalizada", lambda: "MARIA JOSE TORRES" in vendedores["vendedor_clave"].tolist())

[CORRECTO] Catálogo: 60 productos con id_producto único
[CORRECTO] Catálogo: precio_lista numérico
[CORRECTO] Catálogo: marcas sin variantes de mayúsculas (9 marcas)
[CORRECTO] Clientes: 300 únicos
[CORRECTO] Clientes: se conservó la versión más reciente de cada cliente (R8)
[CORRECTO] Clientes: 6 ciudades bien escritas
[CORRECTO] Clientes: sin segmentos nulos
[CORRECTO] Vendedores: comisión como proporción (máximo 0.05)
[CORRECTO] Vendedores: clave normalizada


True

#### Ejercicio 11. Estandarizar el inventario

Implemente `estandarizar_inventario(inv)` que reciba el diccionario del inventario y devuelva un diccionario con las mismas claves donde:

- **R10:** en `stock`, los valores negativos de `stock_sucursal` se reemplazan por nulo (registre cuántos con `logger.warning`). Elimine la columna `fecha_corte`.
- `sucursales` y `costos` se devuelven sin cambios (**R11:** los costos nulos no se imputan).

Use `.copy()` para no modificar los DataFrames originales.

In [46]:
def estandarizar_inventario(inv: dict) -> dict:
    resultado = {nombre: tabla.copy() for nombre, tabla in inv.items()}
    negativos = resultado["stock"]["stock_sucursal"].lt(0)
    logger.warning("R10: %s stocks negativos reemplazados por nulo", int(negativos.sum()))
    resultado["stock"]["stock_sucursal"] = resultado["stock"]["stock_sucursal"].astype("Int64")
    resultado["stock"].loc[negativos, "stock_sucursal"] = pd.NA
    resultado["stock"] = resultado["stock"].drop(columns="fecha_corte")
    return resultado

In [47]:
inventario = ejecutar_paso("Estandarización del inventario", estandarizar_inventario, inventario_raw)

WARNING | R10: 6 stocks negativos reemplazados por nulo


**Verificación del Ejercicio 11**

In [48]:
verificar("Stock sin valores negativos", lambda: (inventario["stock"]["stock_sucursal"].dropna() >= 0).all())
verificar("6 stocks quedaron como nulos", lambda: inventario["stock"]["stock_sucursal"].isna().sum() == 6)
verificar("El original no fue modificado", lambda: (inventario_raw["stock"]["stock_sucursal"] < 0).sum() == 6)
verificar("Costos sin imputar (3 nulos)", lambda: inventario["costos"]["costo_unitario"].isna().sum() == 3)

[CORRECTO] Stock sin valores negativos
[CORRECTO] 6 stocks quedaron como nulos
[CORRECTO] El original no fue modificado
[CORRECTO] Costos sin imputar (3 nulos)


True

### 5.4. Persistencia intermedia

Guardar las fuentes estandarizadas en `data/interim` permite revisar cada una por separado y reanudar el proceso sin repetir la extracción.

In [49]:
def guardar_interim(tablas: dict, config: dict) -> None:
    """Guarda cada fuente estandarizada en data/interim como CSV."""
    destino = config["rutas"]["interim"]
    destino.mkdir(parents=True, exist_ok=True)
    for nombre, df in tablas.items():
        df.to_csv(destino / f"{nombre}.csv", index=False, encoding="utf-8")
    logger.info(f"Guardadas {len(tablas)} tablas intermedias en {destino}")


ejecutar_paso("Persistencia intermedia", guardar_interim, {
    "ventas": ventas, "catalogo": catalogo, "clientes": clientes, "vendedores": vendedores,
    "sucursales": inventario["sucursales"], "costos": inventario["costos"], "stock": inventario["stock"],
}, CONFIG)

INFO | Guardadas 7 tablas intermedias en D:\PUCESA\SEPTIMO_SEMESTRE\CIENCIADEDATOS\LABORATORIOS\ciencia-de-datos\NOTEBOOKS\Clase11_11_2026\data\interim


---
## 6. Integración

Las ventas son la **tabla de hechos**: cada fila es una línea de venta. Las demás fuentes son **tablas de dimensión** que la enriquecen. Por eso todas las uniones son `how="left"` desde ventas.

Dos controles protegen la integración:

- `validate="many_to_one"`: Pandas lanza un error si la tabla de dimensión tiene claves repetidas, lo que **multiplicaría** filas de ventas.
- `indicator=True`: agrega la columna `_merge` que indica si cada fila encontró pareja (`both`) o no (`left_only`), lo que permite detectar **claves huérfanas**.

#### Ejercicio 12. Integrar las fuentes

Implemente `integrar_fuentes(ventas, catalogo, clientes, vendedores, inventario, config)` que devuelva `(df, rechazados)`:

1. **Productos:** una con `catalogo` por `id_producto` (`validate="many_to_one"`, `indicator=True`). **R3:** las ventas sin producto en el catálogo van a `rechazados` con su `motivo` (solo con las columnas de `ventas`).
2. **Clientes:** una con `clientes` por `id_cliente`. Cree `cliente_en_crm` (booleano). Para `C0000` asigne `cliente = "CONSUMIDOR FINAL"` y `segmento = "Consumidor final"` (**R4**). Para los clientes no encontrados que no son `C0000`, asigne `"NO REGISTRADO"` y `"No registrado"` (**R5**).
3. **Vendedores:** elimine la columna original `vendedor` de ventas y una con `vendedores` por `vendedor_clave`. Los no encontrados quedan con `vendedor = "SIN ASIGNAR"` (**R6**).
4. **Inventario:** una con `sucursales` por `id_sucursal`, con `costos` por `id_producto` y con `stock` por `["id_sucursal", "id_producto"]`.
5. Compruebe con `assert` que el número de filas final es igual al número de ventas menos las rechazadas.

In [50]:
def integrar_fuentes(ventas, catalogo, clientes, vendedores, inventario, config):
    n_ventas = len(ventas)
    df = ventas.merge(catalogo, on="id_producto", how="left",
                       validate="many_to_one", indicator=True)
    sin_producto = df["_merge"].eq("left_only")
    rechazados = df.loc[sin_producto, ventas.columns].copy()
    rechazados["motivo"] = "R3: producto inexistente en el catálogo"
    df = df.loc[~sin_producto].drop(columns="_merge")
    df = df.merge(clientes, on="id_cliente", how="left",
                   validate="many_to_one", indicator=True)
    df["cliente_en_crm"] = df["_merge"].eq("both")
    generico = df["id_cliente"].eq(config["cliente_consumidor_final"])
    no_registrado = ~df["cliente_en_crm"] & ~generico
    df.loc[generico, ["cliente", "segmento"]] = ["CONSUMIDOR FINAL", "Consumidor final"]
    df.loc[no_registrado, ["cliente", "segmento"]] = ["NO REGISTRADO", "No registrado"]
    df = df.drop(columns=["_merge", "vendedor"]).merge(
        vendedores, on="vendedor_clave", how="left", validate="many_to_one", indicator=True)
    df["vendedor"] = df["vendedor"].fillna("SIN ASIGNAR")
    df = df.drop(columns="_merge")
    for nombre, claves in [("sucursales", ["id_sucursal"]), ("costos", ["id_producto"]),
                           ("stock", ["id_sucursal", "id_producto"])]:
        df = df.merge(inventario[nombre], on=claves, how="left", validate="many_to_one")
    assert len(df) == n_ventas - len(rechazados), "La integración alteró la cantidad de ventas"
    return df.reset_index(drop=True), rechazados.reset_index(drop=True)

In [51]:
resultado = ejecutar_paso("Integración", integrar_fuentes,
                          ventas, catalogo, clientes, vendedores, inventario, CONFIG)
integrado, rechazados_producto = resultado if resultado else (None, None)
integrado.head() if integrado is not None else None

,id_venta,fecha,cod_producto,id_cliente,cantidad,precio_unitario,descuento,fuente,id_sucursal,id_producto,vendedor_clave,sku,producto,marca,categoria,estado_producto,precio_lista,garantia_meses,peso_kg,cliente,email,telefono,ciudad_cliente,provincia_cliente,segmento,fecha_registro,cliente_en_crm,vendedor,fecha_ingreso,comision_pct,estado_vendedor,sucursal,ciudad_sucursal,provincia_sucursal,costo_unitario,stock_sucursal
0,AMB-00001,2025-01-01,TA-0024,C0216,1,9.09,0.00,ambato,1,24,MARIA JOSE TORRES,TA-0024,Memoria USB HP F92,HP,Almacenamiento,activo,8.87,12.0,0.02,Mateo Vargas Villacís,mateo.vargas216@correo.ec,0987014874,Riobamba,Chimborazo,Hogar,2024-06-29,True,María José Torres,2021-03-15,0.05,Activo,TecnoAndes Ambato,Ambato,Tungurahua,6.81,21
1,AMB-00002,2025-01-01,TA-0025,C0137,1,91.73,0.15,ambato,1,25,SOFIA PAREDES,TA-0025,Disco SSD Sony X48,SONY,Almacenamiento,activo,90.22,12.0,0.06,Andrés Ortiz Paredes,andres.ortiz137@correo.ec,0985024976,Pelileo,Tungurahua,Estudiante,2023-11-23,True,Sofía Paredes,2024-01-10,0.03,Activo,TecnoAndes Ambato,Ambato,Tungurahua,52.14,50
2,AMB-00003,2025-01-02,TA-0031,C0177,2,57.61,0.00,ambato,1,31,SOFIA PAREDES,TA-0031,Switch Samsung G46,SAMSUNG,Redes,descontinuado,57.34,12.0,0.44,Paola Jácome Paredes,paola.jacome177@correo.ec,0985990923,Latacunga,Cotopaxi,Hogar,2024-12-02,True,Sofía Paredes,2024-01-10,0.03,Activo,TecnoAndes Ambato,Ambato,Tungurahua,38.39,43
3,AMB-00004,2025-01-02,TA-0024,C0164,1,9.30,0.00,ambato,1,24,MARIA JOSE TORRES,TA-0024,Memoria USB HP F92,HP,Almacenamiento,activo,8.87,12.0,0.02,María Toapanta Vargas,maria.toapanta164@correo.ec,0983919397,Latacunga,Cotopaxi,Estudiante,2024-05-02,True,María José Torres,2021-03-15,0.05,Activo,TecnoAndes Ambato,Ambato,Tungurahua,6.81,21
4,AMB-00005,2025-01-04,TA-0001,C0017,3,857.95,0.00,ambato,1,1,SOFIA PAREDES,TA-0001,Laptop TP-Link K92,TP-LINK,Computación,activo,876.13,6.0,2.08,Lucía Chávez López,lucia.chavez17@correo.ec,0987137486,Latacunga,Cotopaxi,Empresa,2025-01-07,True,Sofía Paredes,2024-01-10,0.03,Activo,TecnoAndes Ambato,Ambato,Tungurahua,662.18,22


**Verificación del Ejercicio 12**

In [52]:
verificar("1857 filas integradas", lambda: len(integrado) == 1857)
verificar("7 ventas rechazadas por producto inexistente", lambda: len(rechazados_producto) == 7)
verificar("Todas las ventas tienen producto y categoría",
          lambda: integrado[["producto", "categoria"]].notna().all().all())
verificar("Todas las ventas tienen cliente y segmento",
          lambda: integrado[["cliente", "segmento"]].notna().all().all())
verificar("6 ventas con cliente NO REGISTRADO", lambda: (integrado["cliente"] == "NO REGISTRADO").sum() == 6)
verificar("Todas las ventas tienen vendedor; 5 SIN ASIGNAR",
          lambda: integrado["vendedor"].notna().all() and (integrado["vendedor"] == "SIN ASIGNAR").sum() == 5)
verificar("Los nombres de vendedor quedaron en su forma oficial (con tildes)",
          lambda: "María José Torres" in integrado["vendedor"].values and "MARIA JOSE TORRES" not in integrado["vendedor"].values)
verificar("Todas las ventas tienen sucursal", lambda: integrado["sucursal"].notna().all())

[CORRECTO] 1857 filas integradas
[CORRECTO] 7 ventas rechazadas por producto inexistente
[CORRECTO] Todas las ventas tienen producto y categoría
[CORRECTO] Todas las ventas tienen cliente y segmento
[CORRECTO] 6 ventas con cliente NO REGISTRADO
[CORRECTO] Todas las ventas tienen vendedor; 5 SIN ASIGNAR
[CORRECTO] Los nombres de vendedor quedaron en su forma oficial (con tildes)
[CORRECTO] Todas las ventas tienen sucursal


True

**Pregunta.** ¿Qué habría pasado si la unión con vendedores se hubiera hecho con la columna `vendedor` original en lugar de `vendedor_clave`? Compruébelo con código.

In [53]:
ventas_con_producto = ventas.loc[ventas["id_producto"].isin(catalogo["id_producto"])]
union_original = ventas_con_producto.merge(vendedores, on="vendedor", how="left",
                                            validate="many_to_one", indicator=True)
union_normalizada = ventas_con_producto.merge(vendedores, on="vendedor_clave", how="left",
                                               validate="many_to_one", indicator=True)
comparacion_nombres = pd.DataFrame({
    "tipo_union": ["Nombre original", "Clave normalizada"],
    "coincidencias": [union_original["_merge"].eq("both").sum(),
                      union_normalizada["_merge"].eq("both").sum()],
    "sin_coincidencia": [union_original["_merge"].eq("left_only").sum(),
                         union_normalizada["_merge"].eq("left_only").sum()]
})
comparacion_nombres

,tipo_union,coincidencias,sin_coincidencia
0,Nombre original,1028,829
1,Clave normalizada,1852,5


**Respuesta:** La unión por nombre original deja sin correspondencia ventas cuyos vendedores
sí constan en la nómina: las tildes, las mayúsculas y los espacios cambian el texto.
La comparación anterior cuantifica la diferencia. La clave normalizada recupera esas coincidencias
y el nombre final procede de la nómina, por lo que conserva la escritura oficial con tildes.
Los cinco registros sin vendedor se mantienen como `SIN ASIGNAR` (R6).

---
## 7. Variables derivadas y estructura final

El dataset final debe estar **listo para el EDA**: con las métricas de negocio calculadas, columnas de tiempo para agrupar y un orden de columnas lógico.

#### Ejercicio 13. Derivar variables y construir la estructura final

Implemente `construir_dataset_final(df, config)` que:

1. Cree `anio`, `mes`, `trimestre` y `dia_semana` (nombre del día en español; use `dt.dayofweek` y el diccionario `DIAS`).
2. Calcule, redondeando a 2 decimales:
   - `total_bruto = cantidad * precio_unitario`
   - `valor_descuento = total_bruto * descuento`
   - `total_neto = total_bruto - valor_descuento`
   - `costo_total = cantidad * costo_unitario`
   - `margen_bruto = total_neto - costo_total`
3. **R7:** cree `es_venta_atipica` (cantidad mayor que `config["umbral_cantidad_atipica"]`).
4. Seleccione `COLUMNAS_FINALES`, ordene por `fecha` e `id_venta` y reinicie el índice.

In [54]:
COLUMNAS_FINALES = [
    "id_venta", "fecha", "anio", "mes", "trimestre", "dia_semana",
    "id_sucursal", "sucursal", "ciudad_sucursal", "provincia_sucursal",
    "id_producto", "sku", "producto", "marca", "categoria", "estado_producto",
    "cantidad", "precio_lista", "precio_unitario", "descuento",
    "total_bruto", "valor_descuento", "total_neto", "costo_unitario", "costo_total", "margen_bruto",
    "id_cliente", "cliente", "segmento", "ciudad_cliente", "provincia_cliente", "cliente_en_crm",
    "vendedor", "comision_pct", "stock_sucursal", "es_venta_atipica", "fuente",
]

DIAS = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}


def construir_dataset_final(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    df = df.copy()
    df["anio"] = df["fecha"].dt.year
    df["mes"] = df["fecha"].dt.month
    df["trimestre"] = df["fecha"].dt.quarter
    df["dia_semana"] = df["fecha"].dt.dayofweek.map(DIAS)
    df["total_bruto"] = (df["cantidad"].astype("float64") * df["precio_unitario"]).round(2)
    df["valor_descuento"] = (df["total_bruto"] * df["descuento"]).round(2)
    df["total_neto"] = (df["total_bruto"] - df["valor_descuento"]).round(2)
    df["costo_total"] = (df["cantidad"].astype("float64") * df["costo_unitario"]).round(2)
    df["margen_bruto"] = (df["total_neto"] - df["costo_total"]).round(2)
    df["es_venta_atipica"] = df["cantidad"].gt(config["umbral_cantidad_atipica"])
    return df[COLUMNAS_FINALES].sort_values(["fecha", "id_venta"], kind="stable").reset_index(drop=True)

In [55]:
dataset = ejecutar_paso("Construcción del dataset final", construir_dataset_final, integrado, CONFIG)
dataset.head() if dataset is not None else None

,id_venta,fecha,anio,mes,trimestre,dia_semana,id_sucursal,sucursal,ciudad_sucursal,provincia_sucursal,id_producto,sku,producto,marca,categoria,estado_producto,cantidad,precio_lista,precio_unitario,descuento,total_bruto,valor_descuento,total_neto,costo_unitario,costo_total,margen_bruto,id_cliente,cliente,segmento,ciudad_cliente,provincia_cliente,cliente_en_crm,vendedor,comision_pct,stock_sucursal,es_venta_atipica,fuente
0,AMB-00001,2025-01-01,2025,1,1,Miércoles,1,TecnoAndes Ambato,Ambato,Tungurahua,24,TA-0024,Memoria USB HP F92,HP,Almacenamiento,activo,1,8.87,9.09,0.00,9.09,0.00,9.09,6.81,6.81,2.28,C0216,Mateo Vargas Villacís,Hogar,Riobamba,Chimborazo,True,María José Torres,0.05,21,False,ambato
1,AMB-00002,2025-01-01,2025,1,1,Miércoles,1,TecnoAndes Ambato,Ambato,Tungurahua,25,TA-0025,Disco SSD Sony X48,SONY,Almacenamiento,activo,1,90.22,91.73,0.15,91.73,13.76,77.97,52.14,52.14,25.83,C0137,Andrés Ortiz Paredes,Estudiante,Pelileo,Tungurahua,True,Sofía Paredes,0.03,50,False,ambato
2,AMB-00003,2025-01-02,2025,1,1,Jueves,1,TecnoAndes Ambato,Ambato,Tungurahua,31,TA-0031,Switch Samsung G46,SAMSUNG,Redes,descontinuado,2,57.34,57.61,0.00,115.22,0.00,115.22,38.39,76.78,38.44,C0177,Paola Jácome Paredes,Hogar,Latacunga,Cotopaxi,True,Sofía Paredes,0.03,43,False,ambato
3,AMB-00004,2025-01-02,2025,1,1,Jueves,1,TecnoAndes Ambato,Ambato,Tungurahua,24,TA-0024,Memoria USB HP F92,HP,Almacenamiento,activo,1,8.87,9.30,0.00,9.30,0.00,9.30,6.81,6.81,2.49,C0164,María Toapanta Vargas,Estudiante,Latacunga,Cotopaxi,True,María José Torres,0.05,21,False,ambato
4,LAT-00001,2025-01-02,2025,1,1,Jueves,2,TecnoAndes Latacunga,Latacunga,Cotopaxi,60,TA-0060,Escáner Epson G34,EPSON,Impresión,activo,1,136.44,139.51,0.00,139.51,0.00,139.51,101.63,101.63,37.88,C0076,José Paredes Freire,Empresa,Latacunga,Cotopaxi,True,Diego Chávez,0.03,8,False,latacunga


**Verificación del Ejercicio 13**

In [56]:
verificar("El dataset tiene exactamente las columnas finales en orden",
          lambda: list(dataset.columns) == COLUMNAS_FINALES)
verificar("total_neto = total_bruto - valor_descuento",
          lambda: ((dataset["total_bruto"] - dataset["valor_descuento"] - dataset["total_neto"]).abs() < 0.011).all())
verificar("5 ventas atípicas marcadas", lambda: dataset["es_venta_atipica"].sum() == 5)
verificar("Margen nulo solo donde falta el costo",
          lambda: (dataset["margen_bruto"].isna() == dataset["costo_unitario"].isna()).all())
verificar("Ordenado por fecha", lambda: dataset["fecha"].is_monotonic_increasing)
verificar("Total neto vendido = 758374.26", lambda: round(float(dataset["total_neto"].sum()), 2) == 758374.26)

[CORRECTO] El dataset tiene exactamente las columnas finales en orden
[CORRECTO] total_neto = total_bruto - valor_descuento
[CORRECTO] 5 ventas atípicas marcadas
[CORRECTO] Margen nulo solo donde falta el costo
[CORRECTO] Ordenado por fecha
[CORRECTO] Total neto vendido = 758374.26


True

---
## 8. Validación del dataset

Las reglas de validación son el **contrato de calidad** del dataset: si alguna falla, el pipeline lo informa antes de que el equipo de análisis use los datos.

In [57]:
def validar_dataset(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """Evalúa las reglas de calidad del dataset final y devuelve un reporte."""
    claves = ["id_venta", "fecha", "id_sucursal", "id_producto", "id_cliente", "vendedor"]
    reglas = [
        ("id_venta es único", df["id_venta"].is_unique),
        ("Sin nulos en columnas clave", bool(df[claves].notna().all().all())),
        ("Cantidad mayor que cero", bool((df["cantidad"] > 0).all())),
        ("Precio unitario mayor que cero", bool((df["precio_unitario"] > 0).all())),
        ("Descuento entre 0 y 1", bool(df["descuento"].between(0, 1).all())),
        ("Total neto no negativo", bool((df["total_neto"] >= 0).all())),
        ("Fechas dentro del período 2025-01-01 a 2026-06-30",
         bool(df["fecha"].between(config["periodo"]["inicio"], config["periodo"]["fin"]).all())),
        ("Todas las ventas tienen producto del catálogo", bool(df["producto"].notna().all())),
        ("Stock no negativo", bool((df["stock_sucursal"].dropna() >= 0).all())),
    ]
    reporte = pd.DataFrame(reglas, columns=["regla", "cumple"])
    reporte["fecha_validacion"] = datetime.now().strftime("%Y-%m-%d")
    for regla, cumple in reglas:
        (logger.info if cumple else logger.error)(f"Validación: {regla} -> {'OK' if cumple else 'FALLA'}")
    return reporte


reporte_calidad = ejecutar_paso("Validación", validar_dataset, dataset, CONFIG)
reporte_calidad

INFO | Validación: id_venta es único -> OK


INFO | Validación: Sin nulos en columnas clave -> OK


INFO | Validación: Cantidad mayor que cero -> OK


INFO | Validación: Precio unitario mayor que cero -> OK


INFO | Validación: Descuento entre 0 y 1 -> OK


INFO | Validación: Total neto no negativo -> OK


INFO | Validación: Fechas dentro del período 2025-01-01 a 2026-06-30 -> OK


INFO | Validación: Todas las ventas tienen producto del catálogo -> OK


INFO | Validación: Stock no negativo -> OK


,regla,cumple,fecha_validacion
0,id_venta es único,True,2026-11-10
1,Sin nulos en columnas clave,True,2026-11-10
2,Cantidad mayor que cero,True,2026-11-10
3,Precio unitario mayor que cero,True,2026-11-10
4,Descuento entre 0 y 1,True,2026-11-10
5,Total neto no negativo,True,2026-11-10
6,Fechas dentro del período 2025-01-01 a 2026-06-30,True,2026-11-10
7,Todas las ventas tienen producto del catálogo,True,2026-11-10
8,Stock no negativo,True,2026-11-10


In [58]:
print("Valores nulos que quedan en el dataset (se tratarán en el EDA):")
dataset.isnull().sum()[lambda s: s > 0]

Valores nulos que quedan en el dataset (se tratarán en el EDA):


costo_unitario       111
costo_total          111
margen_bruto         111
ciudad_cliente        21
provincia_cliente     21
comision_pct           5
stock_sucursal        74
dtype: int64

**Pregunta.** El dataset todavía tiene valores nulos en `costo_unitario`, `costo_total`, `margen_bruto`, `stock_sucursal`, `ciudad_cliente`, `provincia_cliente` y `comision_pct`. ¿Por qué estos nulos **no** son un error del pipeline? ¿Qué debería decidirse en el EDA?

**Respuesta:** Los costos faltan en la fuente y R11 prohíbe imputarlos; por eso también
quedan nulos el costo total y el margen. El stock negativo se convierte en nulo por R10.
Los clientes genéricos o no registrados carecen de ubicación y los vendedores sin asignar
no tienen una comisión conocida. No se inventan esos datos ni se eliminan ventas válidas.

En el EDA se debe medir su cobertura y decidir si se recupera información de las fuentes,
se excluyen solo esos casos de cálculos específicos o se justifica alguna imputación.
El margen agregado debe informar cuántas ventas tienen costo conocido: sumar márgenes
ignorando nulos no representa el margen de todas las ventas. Además, el stock y los costos
son del corte de bodega, no valores históricos de cada fecha de venta.

---
## 9. Carga de resultados

### 9.1. Diccionario de datos

El diccionario describe cada columna del dataset. Es lo primero que consulta quien recibe los datos.

In [59]:
DESCRIPCIONES = {
    "id_venta": "Identificador único de la línea de venta (prefijo de sucursal + número)",
    "fecha": "Fecha de la venta",
    "anio": "Año de la venta", "mes": "Mes de la venta (1-12)", "trimestre": "Trimestre (1-4)",
    "dia_semana": "Día de la semana en español",
    "id_sucursal": "Identificador de la sucursal", "sucursal": "Nombre de la sucursal",
    "ciudad_sucursal": "Ciudad de la sucursal", "provincia_sucursal": "Provincia de la sucursal",
    "id_producto": "Identificador numérico del producto", "sku": "Código del producto en el ERP",
    "producto": "Nombre del producto", "marca": "Marca en mayúsculas", "categoria": "Categoría del producto",
    "estado_producto": "Estado en el catálogo (activo / descontinuado)",
    "cantidad": "Unidades vendidas", "precio_lista": "Precio de catálogo (USD)",
    "precio_unitario": "Precio cobrado por unidad (USD)", "descuento": "Descuento aplicado (proporción 0-1)",
    "total_bruto": "cantidad x precio_unitario (USD)", "valor_descuento": "total_bruto x descuento (USD)",
    "total_neto": "total_bruto - valor_descuento (USD)", "costo_unitario": "Costo por unidad según bodega (USD); nulo si no existe (R11)",
    "costo_total": "cantidad x costo_unitario (USD)", "margen_bruto": "total_neto - costo_total (USD)",
    "id_cliente": "Identificador del cliente; C0000 = consumidor final (R4)", "cliente": "Nombre del cliente",
    "segmento": "Segmento comercial del cliente", "ciudad_cliente": "Ciudad del cliente",
    "provincia_cliente": "Provincia del cliente", "cliente_en_crm": "True si el cliente existe en el CRM",
    "vendedor": "Nombre oficial del vendedor (nómina)", "comision_pct": "Comisión del vendedor (proporción)",
    "stock_sucursal": "Stock del producto en la sucursal al último corte; nulo si era negativo (R10)",
    "es_venta_atipica": "True si la cantidad supera el umbral de venta atípica (R7)",
    "fuente": "Archivo de origen de la venta",
}


def crear_diccionario(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "descripcion": [DESCRIPCIONES.get(c, "") for c in df.columns],
        "nulos": df.isnull().sum().values,
        "ejemplo": df.iloc[0].astype(str).values,
    })


crear_diccionario(dataset)

,columna,tipo,descripcion,nulos,ejemplo
0,id_venta,str,Identificador único de la línea de venta (pref...,0,AMB-00001
1,fecha,datetime64[us],Fecha de la venta,0,2025-01-01 00:00:00
2,anio,int32,Año de la venta,0,2025
3,mes,int32,Mes de la venta (1-12),0,1
4,trimestre,int32,Trimestre (1-4),0,1
5,dia_semana,str,Día de la semana en español,0,Miércoles
6,id_sucursal,int64,Identificador de la sucursal,0,1
7,sucursal,str,Nombre de la sucursal,0,TecnoAndes Ambato
8,ciudad_sucursal,str,Ciudad de la sucursal,0,Ambato
9,provincia_sucursal,str,Provincia de la sucursal,0,Tungurahua


### 9.2. Guardado de los entregables

#### Ejercicio 14. Guardar los resultados

Se guardan en UTF-8 y sin índice el dataset, el diccionario, el reporte de calidad y
los rechazados con su motivo. Parquet queda excluido por indicación del docente.

In [60]:
def guardar_resultados(dataset, rechazados, reporte, config) -> dict:
    destino = config["rutas"]["processed"]
    destino.mkdir(parents=True, exist_ok=True)
    tablas = {
        "dataset_csv": (config["salida"]["nombre"] + ".csv", dataset),
        "diccionario": ("diccionario_datos.csv", crear_diccionario(dataset)),
        "reporte": ("reporte_calidad.csv", reporte),
        "rechazados": ("registros_rechazados.csv", rechazados),
    }
    rutas = {}
    for clave, (nombre, tabla) in tablas.items():
        ruta = destino / nombre
        tabla.to_csv(ruta, index=False, encoding="utf-8", date_format="%Y-%m-%d")
        rutas[clave] = ruta
    return rutas

In [61]:
rechazados = pd.concat([rechazados_cantidad, rechazados_producto], ignore_index=True) \
    if rechazados_cantidad is not None and rechazados_producto is not None else None
rutas_salida = ejecutar_paso("Guardado", guardar_resultados, dataset, rechazados, reporte_calidad, CONFIG)
rutas_salida

{'dataset_csv': WindowsPath('D:/PUCESA/SEPTIMO_SEMESTRE/CIENCIADEDATOS/LABORATORIOS/ciencia-de-datos/NOTEBOOKS/Clase11_11_2026/data/processed/ventas_consolidadas.csv'),
 'diccionario': WindowsPath('D:/PUCESA/SEPTIMO_SEMESTRE/CIENCIADEDATOS/LABORATORIOS/ciencia-de-datos/NOTEBOOKS/Clase11_11_2026/data/processed/diccionario_datos.csv'),
 'reporte': WindowsPath('D:/PUCESA/SEPTIMO_SEMESTRE/CIENCIADEDATOS/LABORATORIOS/ciencia-de-datos/NOTEBOOKS/Clase11_11_2026/data/processed/reporte_calidad.csv'),
 'rechazados': WindowsPath('D:/PUCESA/SEPTIMO_SEMESTRE/CIENCIADEDATOS/LABORATORIOS/ciencia-de-datos/NOTEBOOKS/Clase11_11_2026/data/processed/registros_rechazados.csv')}

**Verificación del Ejercicio 14**

In [62]:
verificar("Existe data/processed/ventas_consolidadas.csv",
          lambda: (CONFIG["rutas"]["processed"] / "ventas_consolidadas.csv").exists())
verificar("El CSV guardado tiene 1857 filas",
          lambda: len(pd.read_csv(CONFIG["rutas"]["processed"] / "ventas_consolidadas.csv")) == 1857)
verificar("Existen diccionario, reporte y rechazados",
          lambda: all((CONFIG["rutas"]["processed"] / a).exists()
                      for a in ["diccionario_datos.csv", "reporte_calidad.csv", "registros_rechazados.csv"]))
verificar("Se guardaron 23 registros rechazados con su motivo",
          lambda: pd.read_csv(CONFIG["rutas"]["processed"] / "registros_rechazados.csv")["motivo"].notna().sum() == 23)

[CORRECTO] Existe data/processed/ventas_consolidadas.csv


[CORRECTO] El CSV guardado tiene 1857 filas
[CORRECTO] Existen diccionario, reporte y rechazados
[CORRECTO] Se guardaron 23 registros rechazados con su motivo


True

---
## 10. Pipeline reproducible

Hasta aquí cada etapa se ejecutó paso a paso para entenderla. En un entorno real el proceso completo se ejecuta con **una sola instrucción**, por ejemplo cada mes cuando llegan los archivos nuevos.

Un pipeline es **reproducible** cuando, con los mismos datos de entrada y la misma configuración, produce **exactamente** el mismo resultado. Para comprobarlo se compara la **huella digital** (hash MD5) del archivo generado en dos ejecuciones distintas.

#### Ejercicio 15. Orquestar el pipeline

Implemente `ejecutar_pipeline(config)` que ejecute, en orden, todas las etapas usando las funciones ya construidas:

1. Extraer: ventas, catálogo, clientes, vendedores e inventario.
2. Estandarizar: las cinco fuentes y guardar las tablas intermedias.
3. Integrar.
4. Construir el dataset final.
5. Validar. Si alguna regla no se cumple, lance un `ValueError`.
6. Guardar los resultados (los rechazados son la concatenación de los dos grupos de rechazados).

Debe registrar el inicio y el fin con `logger.info` y devolver un diccionario con `dataset`, `rechazados`, `reporte` y `rutas`.

In [63]:
def ejecutar_pipeline(config: dict) -> dict:
    logger.info("INICIO DEL PIPELINE")
    vr = extraer_ventas(config)
    cr = extraer_catalogo(config)
    clr = extraer_clientes(config)
    ver = extraer_vendedores(config)
    ir = extraer_inventario(config)
    v, rech_cantidad = estandarizar_ventas(vr, config)
    c = estandarizar_catalogo(cr)
    cl = estandarizar_clientes(clr)
    ve = estandarizar_vendedores(ver)
    inv = estandarizar_inventario(ir)
    guardar_interim({"ventas": v, "catalogo": c, "clientes": cl, "vendedores": ve, **inv}, config)
    integrado, rech_producto = integrar_fuentes(v, c, cl, ve, inv, config)
    final = construir_dataset_final(integrado, config)
    reporte = validar_dataset(final, config)
    if not reporte["cumple"].all():
        raise ValueError("El dataset incumple reglas de calidad; no se publica el resultado")
    rechazados = pd.concat([rech_cantidad, rech_producto], ignore_index=True)
    assert len(vr) == vr.duplicated().sum() + len(rechazados) + len(final)
    rutas = guardar_resultados(final, rechazados, reporte, config)
    logger.info("FIN DEL PIPELINE: %s ventas y %s rechazados", len(final), len(rechazados))
    return {"dataset": final, "rechazados": rechazados, "reporte": reporte, "rutas": rutas}

### 10.1. Ejecución completa y prueba de reproducibilidad

Se ejecuta el pipeline dos veces y se comparan las huellas de todos los CSV generados.
Cada ejecución sobrescribe sus propios entregables; no se eliminan carpetas.

In [64]:
def huella(ruta: Path) -> str:
    return hashlib.md5(ruta.read_bytes()).hexdigest()


huellas = []
huellas_archivos = []
huellas_raw_antes = {str(p.relative_to(RAW_DIR)): huella(p)
                     for p in sorted(RAW_DIR.rglob("*")) if p.is_file()}
for ejecucion in (1, 2):
    resultado_pipeline = ejecutar_paso(f"Pipeline (ejecución {ejecucion})", ejecutar_pipeline, CONFIG)
    huellas.append(huella(resultado_pipeline["rutas"]["dataset_csv"]))
    archivos = list(resultado_pipeline["rutas"].values()) + sorted(CONFIG["rutas"]["interim"].glob("*.csv"))
    huellas_archivos.append({str(p.relative_to(BASE_DIR)): huella(p) for p in archivos})
for i, h in enumerate(huellas, start=1):
    print(f"Ejecución {i}: {h}")
assert huellas_archivos[0] == huellas_archivos[1], "Cambió algún entregable entre ejecuciones"
assert huellas_raw_antes == {str(p.relative_to(RAW_DIR)): huella(p)
                            for p in sorted(RAW_DIR.rglob("*")) if p.is_file()}
print("Todos los CSV son reproducibles en ambas ejecuciones; fuentes originales intactas.")

INFO | INICIO DEL PIPELINE


INFO | R1: 34 duplicados eliminados


INFO | R2: 16 líneas rechazadas por cantidad


WARNING | R10: 6 stocks negativos reemplazados por nulo


INFO | Guardadas 7 tablas intermedias en D:\PUCESA\SEPTIMO_SEMESTRE\CIENCIADEDATOS\LABORATORIOS\ciencia-de-datos\NOTEBOOKS\Clase11_11_2026\data\interim


INFO | Validación: id_venta es único -> OK


... (21 outputs left out)

INFO | Validación: Todas las ventas tienen producto del catálogo -> OK


INFO | Validación: Stock no negativo -> OK


INFO | FIN DEL PIPELINE: 1857 ventas y 23 rechazados


Ejecución 1: d0f2e91ab7a0646c85f24e6fb3c77ecd
Ejecución 2: d0f2e91ab7a0646c85f24e6fb3c77ecd
Todos los CSV son reproducibles en ambas ejecuciones; fuentes originales intactas.


**Verificación del Ejercicio 15**

In [65]:
verificar("El pipeline se ejecutó dos veces", lambda: len(huellas) == 2)
verificar("Las dos ejecuciones producen exactamente el mismo archivo", lambda: huellas[0] == huellas[1])
verificar("El resultado del pipeline coincide con el construido paso a paso",
          lambda: resultado_pipeline["dataset"].equals(dataset))

[CORRECTO] El pipeline se ejecutó dos veces
[CORRECTO] Las dos ejecuciones producen exactamente el mismo archivo
[CORRECTO] El resultado del pipeline coincide con el construido paso a paso


True

In [66]:
print("Ejecución completada sin generar pipeline.log ni archivos Parquet.")

Ejecución completada sin generar pipeline.log ni archivos Parquet.


---
## 11. Verificación analítica del resultado

Esta sección **no es el EDA**. Su propósito es comprobar que los números del dataset final son coherentes con el negocio antes de entregarlo: si las cifras no tienen sentido, el error está en el pipeline.

### 11.1. Trazabilidad de registros

Todo registro leído debe estar en el dataset final, en los rechazados o haber sido eliminado como duplicado.

In [67]:
final = resultado_pipeline["dataset"] if resultado_pipeline else dataset
rech = resultado_pipeline["rechazados"] if resultado_pipeline else rechazados
duplicados = len(ventas_raw) - len(ventas_raw.drop_duplicates())

trazabilidad = pd.DataFrame({
    "concepto": ["Filas leídas de las 3 sucursales", "(-) Duplicados eliminados (R1)",
                 "(-) Rechazados por cantidad (R2)", "(-) Rechazados por producto (R3)",
                 "(=) Filas en el dataset final"],
    "filas": [len(ventas_raw), -duplicados,
              -(rech["motivo"].str.startswith("R2")).sum(), -(rech["motivo"].str.startswith("R3")).sum(),
              len(final)],
})
display(trazabilidad)
verificar("La trazabilidad cuadra", lambda: trazabilidad["filas"][:4].sum() == trazabilidad["filas"][4])

,concepto,filas
0,Filas leídas de las 3 sucursales,1914
1,(-) Duplicados eliminados (R1),-34
2,(-) Rechazados por cantidad (R2),-16
3,(-) Rechazados por producto (R3),-7
4,(=) Filas en el dataset final,1857


[CORRECTO] La trazabilidad cuadra


True

### 11.2. Cifras de control

In [68]:
resumen_sucursal = final.groupby("sucursal").agg(
    lineas=("id_venta", "count"), unidades=("cantidad", "sum"),
    total_neto=("total_neto", "sum"),
    lineas_con_costo=("costo_unitario", "count"),
    margen_con_costo=("margen_bruto", lambda s: s.sum(min_count=1)),
).round(2)
neto_con_costo = final.loc[final["costo_unitario"].notna()].groupby("sucursal")["total_neto"].sum()
resumen_sucursal["cobertura_costo_pct"] = (100 * resumen_sucursal["lineas_con_costo"] / resumen_sucursal["lineas"]).round(1)
resumen_sucursal["margen_pct_con_costo"] = (100 * resumen_sucursal["margen_con_costo"] / neto_con_costo).round(1)
resumen_sucursal

,lineas,unidades,total_neto,lineas_con_costo,margen_con_costo,cobertura_costo_pct,margen_pct_con_costo
sucursal,,,,,,,
TecnoAndes Ambato,710,1442,306327.95,664,63414.97,93.5,29.4
TecnoAndes Latacunga,633,1194,223874.28,593,56199.23,93.7,29.7
TecnoAndes Píllaro,514,1116,228172.03,489,64700.95,95.1,31.8


In [69]:
pd.pivot_table(final, index="anio", columns="sucursal", values="total_neto",
               aggfunc="sum", margins=True, margins_name="Total").round(2)

sucursal,TecnoAndes Ambato,TecnoAndes Latacunga,TecnoAndes Píllaro,Total
anio,,,,
2025,174239.73,142266.78,170185.55,486692.06
2026,132088.22,81607.50,57986.48,271682.20
Total,306327.95,223874.28,228172.03,758374.26


In [70]:
print("Distribución de ventas por tipo de cliente:")
print(final["segmento"].value_counts())
print("\nVentas atípicas (R7):")
final.loc[final["es_venta_atipica"], ["id_venta", "fecha", "sucursal", "producto", "cantidad", "total_neto"]]

Distribución de ventas por tipo de cliente:
segmento
Empresa             700
Estudiante          562
Hogar               518
No informado         56
Consumidor final     15
No registrado         6
Name: count, dtype: int64

Ventas atípicas (R7):


,id_venta,fecha,sucursal,producto,cantidad,total_neto
47,LAT-00016,2025-01-14,TecnoAndes Latacunga,Teclado Samsung X44,60,4033.20
544,AMB-00205,2025-06-03,TecnoAndes Ambato,Repetidor WiFi Dell B74,85,3554.70
1179,PIL-00317,2025-12-01,TecnoAndes Píllaro,Tablet Kingston K91,120,52530.12
1316,AMB-00516,2026-01-06,TecnoAndes Ambato,Laptop Kingston M72,60,45850.02
1376,PIL-00383,2026-01-24,TecnoAndes Píllaro,Switch Samsung G46,60,3368.40


#### Ejercicio 16. Lectura del dataset desde disco

El equipo de análisis no usará las variables de este notebook, sino el **archivo guardado**. Escriba el código que lea `data/processed/ventas_consolidadas.csv` tal como lo haría el analista (convirtiendo `fecha` a fecha con `parse_dates`) y compruebe con `info()` que los tipos son adecuados para el EDA.

In [71]:
tipos_lectura = {c: "Int64" for c in ["anio", "mes", "trimestre", "id_sucursal", "id_producto",
                                     "cantidad", "stock_sucursal"]}
tipos_lectura.update({c: "string" for c in ["id_venta", "id_cliente", "sku", "producto", "marca",
    "categoria", "estado_producto", "sucursal", "ciudad_sucursal", "provincia_sucursal",
    "cliente", "segmento", "ciudad_cliente", "provincia_cliente", "vendedor", "dia_semana", "fuente"]})
tipos_lectura.update({"cliente_en_crm": "boolean", "es_venta_atipica": "boolean"})
dataset_disco = pd.read_csv(CONFIG["rutas"]["processed"] / "ventas_consolidadas.csv",
                            encoding="utf-8", parse_dates=["fecha"], dtype=tipos_lectura)
dataset_disco.info()
assert pd.api.types.is_datetime64_any_dtype(dataset_disco["fecha"])
assert list(dataset_disco.columns) == COLUMNAS_FINALES
pd.testing.assert_frame_equal(dataset_disco, final, check_dtype=False, check_exact=False,
                              rtol=1e-12, atol=1e-9)
print("El CSV conserva las 1857 ventas, las columnas y sus valores; fecha y tipos restaurados.")

<class 'pandas.DataFrame'>
RangeIndex: 1857 entries, 0 to 1856
Data columns (total 37 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_venta            1857 non-null   string        
 1   fecha               1857 non-null   datetime64[us]
 2   anio                1857 non-null   Int64         
 3   mes                 1857 non-null   Int64         
 4   trimestre           1857 non-null   Int64         
 5   dia_semana          1857 non-null   string        
 6   id_sucursal         1857 non-null   Int64         
 7   sucursal            1857 non-null   string        
 8   ciudad_sucursal     1857 non-null   string        
 9   provincia_sucursal  1857 non-null   string        
 10  id_producto         1857 non-null   Int64         
 11  sku                 1857 non-null   string        
 12  producto            1857 non-null   string        
 13  marca               1857 non-null   string        
 14  cat

---
## 12. Cierre

### 12.1. Decisiones principales

| Decisión | Alternativa descartada | Justificación |
|---|---|---|
| Partir de ventas y usar uniones izquierdas con `many_to_one` | Unir por coincidencias internas o aceptar dimensiones repetidas | Mantiene la unidad de análisis y evita perder o multiplicar ventas; solo R2 y R3 rechazan líneas. |
| Normalizar claves y resolver versiones antes de integrar | Unir nombres sin normalizar o conservar todas las versiones | Recupera coincidencias y aplica R8 al CRM y R9 al catálogo; la nómina aporta el nombre oficial. |
| Conservar nulos justificables y guardar los rechazados con motivo | Rellenar costos con cero o borrar problemas sin evidencia | R10 y R11 evitan datos inventados; R12 mantiene los registros rechazados para revisión. |

### 12.2. Preguntas de reflexión

1. ¿Por qué leer ventas con `dtype=str`?
2. ¿Qué riesgo tiene unir por nombre propio y cómo eliminarlo?
3. ¿Para qué sirven `validate="many_to_one"` e `indicator=True`?
4. ¿Por qué guardar los rechazados?
5. ¿Cómo agregar una cuarta sucursal?
6. ¿Qué demuestra la reproducibilidad y qué puede romperla?
7. ¿Qué queda pendiente para el EDA?

**Respuestas:**

1. Leer los datos como texto permite conservar los códigos originales y después convertir las fechas, los precios y los descuentos según el formato de cada sucursal, es decir, se puede adaptar.
2. Un nombre puede escribirse con diferentes espacios, tildes o mayúsculas. Al unificar su escritura se pueden relacionar mejor las ventas con los vendedores. Sin embargo, dos personas pueden tener el mismo nombre, por lo que la empresa debería registrar un código único para cada empleado.
3. many_to_one comprueba que un código no esté repetido en la tabla que se une a las ventas. Así se evita que una venta aparezca varias veces y aumenten los totales por error. indicator=True permite identificar cuáles registros encontraron una coincidencia y cuáles no.
4. Los registros rechazados se guardan con su motivo para poder revisarlos y corregir los datos de origen, como indica R12. Las cantidades cuadran: 1914 filas leídas = 34 duplicados + 16 rechazadas por cantidad + 7 por producto + 1857 ventas finales.
5. Se agrega Riobamba a fuentes_ventas, indicando su archivo, código de sucursal, columnas y formatos. También deben estar disponibles sus productos, vendedores y existencias. Como la sucursal ya está registrada en la base de datos, se pueden reutilizar las funciones del proceso. Si cambian el período o las cantidades esperadas, también se actualizan sus comprobaciones.
6. Las dos ejecuciones generan archivos con las mismas huellas digitales, lo que demuestra que su contenido es idéntico. El resultado también coincide con el obtenido paso a paso y los archivos originales permanecen intactos. El resultado podría cambiar si se modifican los datos, la configuración, las versiones de las bibliotecas o los redondeos. El reporte de calidad puede tener otra fecha si se ejecuta un día diferente.
7. Falta revisar los datos ausentes de costos, existencias, ubicación y comisiones, y decidir cómo tratarlos en cada análisis. También deben estudiarse las cinco ventas atípicas y cómo pueden influir los clientes no registrados. Al presentar las ganancias, se debe aclarar cuántas ventas tienen costos conocidos. Además, el último registro de inventario no permite conocer las existencias ni los costos que había cuando se realizó cada venta.